# Defter 9 — Bağımsız doğrulama, belirsizlik ve uygulama dosyaları
**Embedding tree growth theory into AI** · Prof. Dr. İlker Ercanlı

1. **I1 Bağımsız duyarlılık testi (ana analiz):** 10° blokların 5 katı sırayla dışarıda bırakılır; U+S ve T+S (3 tohum), ayarlı XGBoost ve karma model eğitilir.
   Dışarıda bırakılan alanlarda gözlenen halkalara ve model tahminlerine **aynı** detrend + kronoloji + iklim regresyonu uygulanır (otokorelasyon düzeltmeli).
2. **I2** Tablo 2 (senaryo tepkileri) için alan bootstrap güven aralıkları.
3. **I3** Toplanmış ölçekte R² (halka, ağaç × on yıl, seri, kronoloji, alan, alan içi yıllar arası).
4. **I4** Projeksiyonlarda tohum (topluluk) belirsizliği + yağış oranı sınırı kontrolü.
5. **I5** B_max kontrolü (kaç ağaç B_max'ın üstünde).
6. **I6** GitHub uygulaması için dosyalar (`ITRDB_AI/UYGULAMA_VERI`): ham halka ölçümü içermez.

**Çalışma zamanı:** T4/L4 GPU, "Yüksek RAM". Süre ≈ 6–7 saat (I1 eğitimleri ≈ 5 saat). Defter 8'in CMIP6 dosyaları ve son modelleri kullanılır, yeniden indirilmez.
Kesilirse sekmeyi kapatıp yeniden açın ve "Tümünü çalıştır": biten eğitimler ve tablolar atlanır.

In [ ]:
# --- Kurulum ---
import os, json, time, math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ANA = '/content/drive/MyDrive/ITRDB_AI'
except ImportError:
    ANA = os.environ.get('ITRDB_AI', '.')
SONUC = os.path.join(ANA, 'SONUC_SON'); os.makedirs(SONUC, exist_ok=True)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Cihaz:', DEV, torch.cuda.get_device_name(0) if DEV == 'cuda' else '(GPU yok: Çalışma zamanı → T4 GPU seçin)')

AYAR = dict(batch=32768, max_epoch=25, sabir=3, lr=2e-3, lr_gizil=1e-2, J=8, gizli=128, emb=8,
            takson_dusur=0.10, su_onsel=0.5, tau_onsel=0.5, sv_onsel=0.5, desc_gurultu=0.2, wd=1e-4,
            rastgele_etki=False,   # False: popülasyon modeli (ana analiz); True: ağaç/alan rastgele etkili duyarlılık analizi
            r0_bas_cm=2.0, sifir_mm=0.005, hizli_deneme=False)

In [ ]:
# --- Veri ---
d = np.load(os.path.join(ANA, 'TENSOR', 'veri_v2.npz'))
soz = json.load(open(os.path.join(ANA, 'TENSOR', 'veri_v2_sozluk.json')))
YIL0 = soz['yil0']; S = len(soz['site_code']); NG = len(soz['genus']); NS = len(soz['species'])
NSER = len(d['series_site']); NTREE = int(d['series_tree'].max()) + 1
t = lambda x, dt=None: torch.as_tensor(np.asarray(x), dtype=dt).to(DEV)

# Halkalar: 1901 halkaları çıkar (önceki yılın iklimi yok)
k = d['ring_year'] > YIL0
R_ser = t(d['ring_series'][k], torch.long); R_yil = t(d['ring_year'][k], torch.long)
R_w = t(d['ring_rw'][k], torch.float32); R_rp = t(d['ring_rprev'][k], torch.float32) / 10   # cm
ser_site = t(d['series_site'], torch.long); ser_tree = t(d['series_tree'], torch.long)
ser_zaman = t(d['series_zaman'], torch.bool)
site_genus = t(d['site_genus'], torch.long); site_species = t(d['site_species'], torch.long)
NR = len(R_w)

# Aylık iklim özellikleri [alan, ay(124*12), kanal]: T, log(1+P), M, VPD, gE
clim = d['clim']; soil = d['soil']; gE = d['gE']
NY = clim.shape[1]
ozel = np.stack([clim[..., 0], np.log1p(np.maximum(clim[..., 1], 0)), soil, clim[..., 3],
                 np.broadcast_to(gE[:, None, :], soil.shape)], -1).reshape(S, NY * 12, 5).astype(np.float32)
OZ = t(ozel); del ozel, clim
oz_ort = OZ.reshape(-1, 5).mean(0); oz_ss = OZ.reshape(-1, 5).std(0)
desc = d['desc']; DESC = t((desc - desc.mean(0)) / desc.std(0), torch.float32)
# Yetişme ortamı: 11 sürekli değişken (standartlaştırılmış) + biyom, ekolojik bölge, Köppen sınıfları (son indeks = bilinmeyen)
env = d['env']; ENV = t((env - env.mean(0)) / env.std(0), torch.float32)
site_biyom = t(d['site_biyom'], torch.long); site_eko = t(d['site_eko'], torch.long); site_kg = t(d['site_kg'], torch.long)
NB, NE, NK = soz['n_biyom'], soz['n_eko'], soz['n_kg']
KAT_GOR = {}   # bölünmeye göre eğitimde görülen sınıflar (görülmeyenler 'bilinmeyen' gömmesine gider)
ORTAM_BOYUT = ENV.shape[1] + 3 * 4

class OrtamGirdi(nn.Module):
    """Alan başına yetişme ortamı vektörü: sürekli değişkenler + biyom/ekolojik bölge/Köppen gömmeleri (her biri 4 boyut)"""
    def __init__(self):
        super().__init__()
        self.b = nn.Embedding(NB + 1, 4); self.e = nn.Embedding(NE + 1, 4); self.k = nn.Embedding(NK + 1, 4)
    def forward(self, G):
        b, e, k = G['b'], G['e'], G['k']
        if KAT_GOR:
            b = torch.where(KAT_GOR['b'][b], b, torch.full_like(b, NB)); e = torch.where(KAT_GOR['e'][e], e, torch.full_like(e, NE))
            k = torch.where(KAT_GOR['k'][k], k, torch.full_like(k, NK))
        if self.training:      # %10 rastgele maskeleme: 'bilinmeyen' gömmesi de öğrenilsin
            m = torch.rand(len(b), device=DEV) < 0.10
            b = torch.where(m, torch.full_like(b, NB), b); e = torch.where(m, torch.full_like(e, NE), e); k = torch.where(m, torch.full_like(k, NK), k)
        x = G['env'] + (AYAR['desc_gurultu'] * torch.randn_like(G['env']) if self.training else 0)
        return torch.cat([x, self.b(b), self.e(e), self.k(k)], 1)
print(f'{NR:,} halka, {NSER:,} seri, {NTREE:,} ağaç, {S} alan — veri {"GPU" if DEV=="cuda" else "CPU"} belleğinde')

# Yerel uyum için alan nem normali: 1901–1960 aylık toprak nemi ortalaması (zamansal testte de yalnız eğitim dönemi)
MBAR = OZ[:, :60 * 12, 2].mean(1); MREF = MBAR.mean()
def alan_girdi(s):
    """Alan düzeyi girdiler (SHAP ve ihlal analizinde gruplar halinde değiştirilebilsin diye sözlük)"""
    return dict(g=site_genus[s], sp=site_species[s], desc=DESC[s], env=ENV[s], b=site_biyom[s], e=site_eko[s], k=site_kg[s], mbar=MBAR[s])

def girdiler(idx):
    """Halka düzeyi girdiler: 24 aylık iklim penceresi X, birikmiş yarıçap rp, seri ve alan girdileri"""
    X, s = pencere(idx); G = alan_girdi(s); G.update(X=X, rp=R_rp[idx], ser=R_ser[idx]); return G

def pencere(idx):
    """Halka idx için 24 aylık pencere (önceki yıl Ocak – bu yıl Aralık): [B, 24, 5]"""
    s = ser_site[R_ser[idx]]
    bas = (R_yil[idx] - 1 - YIL0) * 12
    ay = bas[:, None] + torch.arange(24, device=DEV)[None, :]
    return OZ[s[:, None], ay], s

In [ ]:
# --- Test bölünmeleri ---
site_bolge = np.array(soz['site_bolge']); site_kat = d['site_blok_kat']; site_takson = d['site_takson_test']
site_ic = d['site_ic_dogrulama']

def bolunme(ad):
    """Dönüş: (eğitim, doğrulama, test) halka indeksleri ve test alanlarında takson gizlensin mi"""
    site_r = ser_site[R_ser].cpu().numpy(); yil = R_yil.cpu().numpy(); zuy = ser_zaman[R_ser].cpu().numpy()
    gizle = False
    if ad == 'zaman':
        test = zuy & (yil > 1960); havuz = yil <= 1960
    elif ad.startswith('bolge:'):
        tb = np.isin(site_r, np.where(site_bolge == ad.split(':')[1])[0]); test = tb; havuz = ~tb
    elif ad.startswith('blok:'):
        tb = np.isin(site_r, np.where(site_kat == int(ad.split(':')[1]))[0]); test = tb; havuz = ~tb
    elif ad == 'takson':
        tb = np.isin(site_r, np.where(site_takson)[0]); test = tb; havuz = ~tb; gizle = True
    elif ad.startswith('iklim:'):
        # İklim dışa-genelleme: en sıcak (1901–1960 ort. sıcaklık) ya da en kurak (P/PET) %10 alan yalnız testte
        tur = ad.split(':')[1]
        if tur == 'kurak2':          # %10–20 kurak bant test; eğitim yalnız en kurak %20 dışı alanlarla
            x = -desc[:, 5]; q8, q9 = np.quantile(x, [0.8, 0.9])
            test = np.isin(site_r, np.where((x >= q8) & (x < q9))[0]); havuz = np.isin(site_r, np.where(x < q8)[0])
        else:
            x = {'sicak': desc[:, 2], 'kurak': -desc[:, 5], 'nemli': desc[:, 5]}[tur]
            tb = np.isin(site_r, np.where(x >= np.quantile(x, 0.9))[0]); test = tb; havuz = ~tb
    elif ad == 'tum':            # projeksiyon için son modeller: tüm alanlar ve yıllar; test = iç doğrulama alanları
        havuz = np.ones(len(site_r), bool); test = site_ic[site_r].copy()
    else:
        raise ValueError(ad)
    ic = site_ic[site_r]
    egit = np.where(havuz & ~ic)[0]; dog = np.where(havuz & ic)[0]; test = np.where(test)[0]
    return t(egit, torch.long), t(dog, torch.long), t(test, torch.long), gizle

In [ ]:
# --- Modeller ---
class AlanAgi(nn.Module):
    """z_s -> parametreler. Cins/tür gömmeleri + standartlaştırılmış alan tanımlayıcıları."""
    def __init__(self, n_cikis, emb, gizli, sutun=None, ortam=False):
        super().__init__()
        self.ortam = OrtamGirdi() if ortam else None
        self.sutun = list(range(DESC.shape[1])) if sutun is None else sutun   # saf teori: yalnız [enlem, yükselti]
        self.ge = nn.Embedding(NG + 1, emb); self.se = nn.Embedding(NS + 1, emb)   # son indeks = "bilinmeyen takson"
        self.mlp = nn.Sequential(nn.Linear(2 * emb + len(self.sutun) + (ORTAM_BOYUT if ortam else 0), gizli), nn.GELU(),
                                 nn.Linear(gizli, gizli), nn.GELU(), nn.Linear(gizli, n_cikis))
    def forward(self, G, gizle_maske=None):
        g, sp = G['g'], G['sp']
        if gizle_maske is not None:
            g = torch.where(gizle_maske, torch.full_like(g, NG), g); sp = torch.where(gizle_maske, torch.full_like(sp, NS), sp)
        d = G['desc'][:, self.sutun]
        d = d + (AYAR['desc_gurultu'] * torch.randn_like(d) if self.training else 0)
        parcalar = [self.ge(g), self.se(sp), d] + ([self.ortam(G)] if self.ortam is not None else [])
        return self.mlp(torch.cat(parcalar, 1))

def monoton(x, a, c, s):
    """Denklem (4): negatif olmayan lojistik karışımı, 0–1 arası, azalmayan. x:[B,24], a,c,s:[B,J]"""
    w = F.softplus(a) + 1e-4
    z = torch.sigmoid((x[..., None] - c[:, None, :]) / s[:, None, :])
    return (z * w[:, None, :]).sum(-1) / w.sum(-1, keepdim=True)

def alan_gor_bul(seri_gor):
    m = torch.zeros(S, dtype=torch.bool, device=DEV); m[ser_site[seri_gor]] = True
    return m

class Gizil(nn.Module):
    """Seri başına gizil başlangıç yarıçapı r0 = softplus(ρ) ve ağaç etkisi u; hiyerarşik önsel cezası"""
    def __init__(self, r0_bas):
        super().__init__()
        rho0 = math.log(math.expm1(r0_bas))
        self.rho = nn.Parameter(torch.full((NSER,), rho0, device=DEV))
        self.mu_g = nn.Parameter(torch.full((NG + 1,), rho0, device=DEV))
        self.u = nn.Parameter(torch.zeros(NTREE, device=DEV))          # ağaç etkisi
        self.v = nn.Parameter(torch.zeros(S, device=DEV))              # alan etkisi (toprak, sıklık vb. kaydedilmemiş farklar)
        # Önsel ölçekleri sabit (zayıf bilgili) hiperparametreler: MAP ile öğrenilen varyans bileşenleri sıfıra çöker.
        # Yeni ağaç/seri için öngörüde kullanılan yayılım ise eğitilmiş ağaçların GÖRGÜL dağılımından alınır.
        self.su_onsel, self.tau_onsel = AYAR['su_onsel'], AYAR['tau_onsel']
        self.register_buffer('su_gorgul', torch.tensor(AYAR['su_onsel'], device=DEV))
        self.register_buffer('tau_gorgul', torch.tensor(AYAR['tau_onsel'], device=DEV))
        self.sv_onsel = AYAR['sv_onsel']
        self.register_buffer('sv_gorgul', torch.tensor(AYAR['sv_onsel'], device=DEV))
        self.register_buffer('ort_u', torch.tensor(0., device=DEV)); self.register_buffer('ort_v', torch.tensor(0., device=DEV))
        self.register_buffer('ort_d', torch.tensor(0., device=DEV))
    def su(self): return self.su_gorgul
    def tau(self): return self.tau_gorgul
    def gorgul_guncelle(self, seri_gor, agac_gor):
        with torch.no_grad():
            g = site_genus[ser_site]
            # Görgül dağılım: ortalama da alınır (ağın sabit terimi ile rastgele etkilerin ortalaması birbirine kayabilir)
            u, d, v = self.u[agac_gor], (self.rho - self.mu_g[g])[seri_gor], self.v[alan_gor_bul(seri_gor)]
            self.su_gorgul.copy_(u.std().clamp_min(0.02)); self.ort_u.copy_(u.mean())
            self.tau_gorgul.copy_(d.std().clamp_min(0.02)); self.ort_d.copy_(d.mean())
            self.sv_gorgul.copy_(v.std().clamp_min(0.02)); self.ort_v.copy_(v.mean())
    def ceza(self, seri_gor, agac_gor):
        """Hiyerarşik önsel; YALNIZ eğitimde verisi olan seri ve ağaçlar üzerinden"""
        g = site_genus[ser_site]
        d = (self.rho - self.mu_g[g])[seri_gor]; u = self.u[agac_gor]; v = self.v[alan_gor_bul(seri_gor)]
        return ((d ** 2).sum() / (2 * self.tau_onsel ** 2) + (u ** 2).sum() / (2 * self.su_onsel ** 2)
                + (v ** 2).sum() / (2 * self.sv_onsel ** 2))
    def gizil_deger(self, ser, ornek=None):
        """Halka başına ρ ve u. ornek=(seri_gor, agac_gor, gizle): görülmemişler için önselden rastgele çekim"""
        rho, u, v = self.rho[ser], self.u[ser_tree[ser]], self.v[ser_site[ser]]
        if ornek is not None:
            seri_gor, agac_gor, gizle, mu_glob = ornek
            g = site_genus[ser_site[ser]]
            mu = torch.where(gizle, mu_glob, self.mu_g[g]) if gizle is not None else self.mu_g[g]
            rho = torch.where(seri_gor[ser], rho, mu + self.ort_d + self.tau() * torch.randn_like(rho))
            u = torch.where(agac_gor[ser_tree[ser]], u, self.ort_u + self.su() * torch.randn_like(u))
            v = torch.where(alan_gor_bul(seri_gor)[ser_site[ser]], v, self.ort_v + self.sv_gorgul * torch.randn_like(v))
        return rho, u + v

class TeoriModeli(nn.Module):
    def __init__(self, J=8, emb=8, gizli=128, ablasyon=None, r0_bas=2.0, m_bas=2/3, vpd=False, m_sabit=None, saf=False, ortam=False, uyum=False, genis=False, tur=False):
        super().__init__()
        self.J, self.abl = J, ablasyon
        self.uyum = uyum
        self.mgen = 0.3 if genis else 0.1   # nem eşiği merkezlerinin kayma aralığı
        self.m_logit = math.log(m_bas / (1 - m_bas)); self.vpd = vpd; self.m_sabit = m_sabit
        n = 3 + 6 * J + 24 + 1 + 1          # +1: başlangıç yarıçapı r0 (alan ve takson düzeyinde, ağdan)
        if ablasyon == 'abl_min': n += 3
        if vpd: n += 3 * J                 # VPD yanıtı (üçüncü sınırlayıcı faktör)
        if uyum: n += 1                    # yerel uyum katsayısı λ
        self.Dbaz = torch.linspace(0.2, 3.0, J, device=DEV)
        self.ag = AlanAgi(n, emb, gizli, sutun=[] if tur else ([0, 1] if saf else None), ortam=ortam)   # saf: iklim normalleri parametre ağına girmez
        if ablasyon == 'abl_mono':      # monoton yanıt yerine kısıtsız küçük ağ (girdi: x ve alan parametreleri)
            self.serbT = nn.Sequential(nn.Linear(1 + 3 * J, 32), nn.GELU(), nn.Linear(32, 1))
            self.serbM = nn.Sequential(nn.Linear(1 + 3 * J, 32), nn.GELU(), nn.Linear(32, 1))
        self.gz = Gizil(r0_bas)
        self.Tbaz = torch.linspace(-5, 25, J, device=DEV); self.Mbaz = torch.linspace(0.02, 0.6, J, device=DEV)

    def parametreler(self, G, gizle=None):
        if torch.is_tensor(G): G = alan_girdi(G)
        o = self.ag(G, gizle); J = self.J
        p = {}
        p['eta'] = torch.exp(o[:, 0] + math.log(0.7))
        p['Bmax'] = torch.exp(o[:, 1] + math.log(1e4))                         # cm² (başlangıç: r ≈ 56 cm)
        p['m'] = torch.sigmoid(o[:, 2] + self.m_logit)                            # başlangıç varsayılan 2/3
        if self.m_sabit is not None: p['m'] = torch.full_like(p['m'], self.m_sabit)   # profil olabilirlik: m sabit
        if getattr(self, 'm_cins', None) is not None:   # sentetik veri üretimi: m cinse göre dışarıdan; η aynı ortalama büyümeyi korur
            m_yeni = self.m_cins[G['g']]; p['eta'] = p['eta'] * 300.0 ** (p['m'] - m_yeni); p['m'] = m_yeni
        i = 3
        p['aT'], p['cT'], p['sT'] = o[:, i:i+J], self.Tbaz + 5 * torch.tanh(o[:, i+J:i+2*J]), 0.5 + 2 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['aM'], p['cM'], p['sM'] = o[:, i:i+J], self.Mbaz + self.mgen * torch.tanh(o[:, i+J:i+2*J]), 0.005 + 0.03 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['omega'] = torch.softmax(o[:, i:i+24], 1); i += 24
        if self.vpd:
            p['aD'], p['cD'], p['sD'] = o[:, i:i+J], self.Dbaz + 0.3 * torch.tanh(o[:, i+J:i+2*J]), 0.05 + 0.3 * F.softplus(o[:, i+2*J:i+3*J]); i += 3 * J
        p['log_sigma'] = o[:, i] - 0.3; i += 1
        p['r0'] = F.softplus(o[:, i] + math.log(math.expm1(AYAR['r0_bas_cm']))); i += 1   # cm
        if self.abl == 'abl_min': p['karisim'] = torch.softmax(o[:, i:i+3], 1)
        if self.uyum: p['lam'] = torch.sigmoid(o[:, -1])          # 0 = mutlak tepki, 1 = tamamen yerel uyumlu
        return p

    def forward(self, idx, gizle=None, ornek=None, G=None):
        G = girdiler(idx) if G is None else G
        X = G['X']; p = self.parametreler(G, gizle); G0 = G
        ser = G['ser']
        rho_i, u_i = self.gz.gizil_deger(ser, ornek)
        if not AYAR['rastgele_etki']: u_i = torch.zeros_like(u_i)
        r0 = 0.05 if self.abl == 'abl_r0' else (F.softplus(rho_i) if AYAR['rastgele_etki'] else p['r0'])
        r = r0 + G['rp']; B = math.pi * r ** 2                                   # cm²
        # Denklem (2): büyüme potansiyeli
        if self.abl == 'abl_G':
            G = p['eta'] * 30.0
        else:
            x = (B / p['Bmax']).clamp_min(1e-8) ** (1 - p['m'])
            G = p['eta'] * B.clamp_min(1e-6) ** p['m'] * F.softplus(20 * (1 - x)) / 20
        # Denklem (3)–(4): sınırlayıcı faktör
        T, M, gEk = X[..., 0], X[..., 2], X[..., 4]
        if self.uyum: M = M - p['lam'][:, None] * (G0['mbar'][:, None] - MREF)   # eşik alanın nem normaline göre kayar
        if self.abl == 'abl_mono':
            pT = torch.cat([p['aT'], p['cT'], p['sT']], 1); pM = torch.cat([p['aM'], p['cM'], p['sM']], 1)
            gT = torch.sigmoid(self.serbT(torch.cat([T[..., None] / 10, pT[:, None, :].expand(-1, 24, -1)], -1))[..., 0])
            gM = torch.sigmoid(self.serbM(torch.cat([M[..., None] * 5, pM[:, None, :].expand(-1, 24, -1)], -1))[..., 0])
        else:
            gT = monoton(T, p['aT'], p['cT'], p['sT']); gM = monoton(M, p['aM'], p['cM'], p['sM'])
        if self.abl == 'abl_min':
            k = p['karisim'][:, None, :]
            sinir = k[..., 0] * gT + k[..., 1] * gM + k[..., 2] * gT * gM
        else:
            sinir = torch.minimum(gT, gM)
        if self.vpd:     # Liebig: min(g_T, g_M, g_D); g_D = 1 − monoton(VPD) → atmosfer kurudukça büyüme azalır
            gD = 1 - monoton(X[..., 3], p['aD'], p['cD'], p['sD'])
            sinir = torch.minimum(sinir, gD)
        C = (p['omega'] * gEk * sinir).sum(1).clamp_min(1e-6)
        BAI = G * C * torch.exp(u_i)
        w_hat = (torch.sqrt((B + BAI) / math.pi) - r) * 10                         # mm
        mu = torch.log(w_hat.clamp_min(1e-4))
        return mu, p['log_sigma'], dict(gT=gT, gM=gM, C=C, G=G, lam=p.get('lam'))

class SerbestAg(nn.Module):
    """Kısıtsız karşılaştırma: 24 aylık iklim -> zamansal evrişim; + log yarıçap + alan gömmesi -> log w"""
    def __init__(self, emb=8, gizli=40, r0_bas=2.0, ortam=False):
        super().__init__()
        self.ortam = OrtamGirdi() if ortam else None
        self.conv = nn.Sequential(nn.Conv1d(5, 32, 3), nn.GELU(), nn.Conv1d(32, 32, 3), nn.GELU())
        self.ge = nn.Embedding(NG + 1, emb); self.se = nn.Embedding(NS + 1, emb)
        self.mlp = nn.Sequential(nn.Linear(32 * 20 + 2 * emb + DESC.shape[1] + 1 + (ORTAM_BOYUT if ortam else 0), gizli), nn.GELU(),
                                 nn.Linear(gizli, gizli), nn.GELU(), nn.Linear(gizli, 2))
        self.gz = Gizil(r0_bas)
    def forward(self, idx, gizle=None, ornek=None, G=None):
        G = girdiler(idx) if G is None else G
        X = (G['X'] - oz_ort) / oz_ss
        h = self.conv(X.transpose(1, 2)).flatten(1)
        g, sp = G['g'], G['sp']
        if gizle is not None:
            g = torch.where(gizle, torch.full_like(g, NG), g); sp = torch.where(gizle, torch.full_like(sp, NS), sp)
        lr = torch.log1p(G['rp'])[:, None]
        d = G['desc'] + (AYAR['desc_gurultu'] * torch.randn_like(G['desc']) if self.training else 0)
        parcalar = [h, self.ge(g), self.se(sp), d, lr] + ([self.ortam(G)] if self.ortam is not None else [])
        o = self.mlp(torch.cat(parcalar, 1))
        _, u_i = self.gz.gizil_deger(G['ser'], ornek)
        if not AYAR['rastgele_etki']: u_i = torch.zeros_like(u_i)
        return o[:, 0] - 0.3 + u_i, o[:, 1] - 0.3, {}

def yap(ad):
    """Model adı: taban ad + isteğe bağlı '_ortam' eki (yetişme ortamı girdileri)"""
    ortam = ad.endswith('_ortam'); ad = ad[:-6] if ortam else ad
    T = lambda **k: TeoriModeli(AYAR['J'], AYAR['emb'], AYAR['gizli'], k.pop('abl', None), AYAR['r0_bas_cm'], ortam=ortam, **k).to(DEV)
    if ad == 'serbest': return SerbestAg(AYAR['emb'], r0_bas=AYAR['r0_bas_cm'], ortam=ortam).to(DEV)
    if ad == 'teori': return T()
    if ad == 'teori_vpd': return T(vpd=True)
    if ad == 'teori_uyum': return T(uyum=True)
    if ad == 'teori_genis': return T(genis=True)
    if ad == 'teori_tur': return T(tur=True)
    if ad == 'teori_saf': return T(vpd=True, saf=True)
    if ad.startswith('teori_mf'):       # profil olabilirlik: teori_mf050 -> m = 0.50 sabit
        return T(m_sabit=int(ad[8:]) / 1000 if len(ad) == 12 else int(ad[8:]) / 100)
    if ad.startswith('teori_m'): return T(m_bas=int(ad[7:]) / 100)
    if ad.startswith('abl_'): return T(abl=ad)
    raise ValueError(ad)

def ag_param_sayisi(m):
    return sum(p.numel() for n, p in m.named_parameters() if not n.startswith('gz.'))

for ad in ['teori_ortam', 'teori_tur']:
    print(f'{ad:16s} eğitilebilir ağ parametresi: {ag_param_sayisi(yap(ad)):,}')

In [ ]:
# --- Eğitim ---
LOG_Y_SIFIR = math.log(AYAR['sifir_mm'])
def hedef(idx):
    w = R_w[idx]
    return torch.where(w > 0, torch.log(w.clamp_min(1e-6)), torch.full_like(w, LOG_Y_SIFIR))

LOG_ESIK = math.log(0.01)            # ölçüm çözünürlüğü: sıfır halka = "0,01 mm'den küçük" (sansürlü gözlem)
def nll(mu, ls, y):
    sifir = y <= LOG_Y_SIFIR + 1e-6
    tam = 0.5 * ((y - mu) / torch.exp(ls)) ** 2 + ls + 0.9189385
    sans = -torch.special.log_ndtr((LOG_ESIK - mu) / torch.exp(ls))
    return torch.where(sifir, sans, tam).mean()

@torch.no_grad()
def tahmin(model, idx, seri_gor, agac_gor, gizle_alan=False, K=8, bs=131072):
    """Görülmüş ağaçlar: koşullu öngörü. Görülmemiş ağaçlar (yeni alan/tür): ρ ve u önselden K kez çekilir;
    öngörü = karışımın ortalaması, varyans = ort(σ²) + var(μ)  (log ölçeğinde)"""
    model.eval(); MU, SG = [], []
    g_gor = torch.unique(site_genus[ser_site[seri_gor.nonzero().squeeze(1)]])
    mu_glob = model.gz.mu_g[g_gor].mean()
    with torch.no_grad():
        for i in range(0, len(idx), bs):
            b = idx[i:i+bs]
            gz = torch.ones(len(b), dtype=torch.bool, device=DEV) if gizle_alan else None
            yeni = ~agac_gor[ser_tree[R_ser[b]]] | ~seri_gor[R_ser[b]]
            if not AYAR['rastgele_etki']: yeni = torch.zeros_like(yeni)
            if not yeni.any():
                mu, ls, _ = model(b, gz); MU.append(mu); SG.append(torch.exp(ls)); continue
            ms, vs = [], []
            for k in range(K):
                mu, ls, _ = model(b, gz, ornek=(seri_gor, agac_gor, gz, mu_glob)); ms.append(mu); vs.append(torch.exp(2 * ls))
            ms, vs = torch.stack(ms), torch.stack(vs)
            MU.append(ms.mean(0)); SG.append(torch.sqrt(vs.mean(0) + ms.var(0, unbiased=False)))
    model.train()
    return torch.cat(MU), torch.cat(SG)

def kat_gor_ayarla(egt):
    """Eğitim halkalarının alanlarında görülen biyom / ekolojik bölge / Köppen sınıfları"""
    sg = torch.unique(ser_site[R_ser[egt]]); KAT_GOR.clear()
    for anahtar, dizi, n in [('b', site_biyom, NB), ('e', site_eko, NE), ('k', site_kg, NK)]:
        m = torch.zeros(n + 1, dtype=torch.bool, device=DEV); m[dizi[sg]] = True; KAT_GOR[anahtar] = m

def egit(bolunme_ad, model_ad, tohum):
    is_ad = f'{bolunme_ad.replace(":", "-")}__{model_ad}__s{tohum}'
    klasor = os.path.join(SONUC, is_ad); os.makedirs(klasor, exist_ok=True)
    if os.path.exists(os.path.join(klasor, 'bitti.json')):
        print('atlandı (bitmiş):', is_ad); return
    torch.manual_seed(tohum); np.random.seed(tohum)
    egt, dog, test, gizle = bolunme(bolunme_ad)
    kat_gor_ayarla(egt)
    if AYAR['hizli_deneme']:
        egt, dog, test = egt[torch.randperm(len(egt), device=DEV)[:200000]], dog[:20000], test[:20000]
    model = yap(model_ad)
    gizil = [p for n, p in model.named_parameters() if n.startswith('gz.')]
    ag = [p for n, p in model.named_parameters() if not n.startswith('gz.')]
    opt = torch.optim.AdamW([{'params': ag, 'lr': AYAR['lr'], 'weight_decay': AYAR['wd']},
                             {'params': gizil, 'lr': AYAR['lr_gizil'] if AYAR['rastgele_etki'] else 0.0, 'weight_decay': 0.0}])
    kayit = os.path.join(klasor, 'son.pt'); bas_ep, en_iyi, sabir, gecmis = 0, float('inf'), 0, []
    if os.path.exists(kayit):
        ck = torch.load(kayit, map_location=DEV)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        bas_ep, en_iyi, sabir, gecmis = ck['epoch'] + 1, ck['en_iyi'], ck['sabir'], ck['gecmis']
        print(f'{is_ad}: {bas_ep}. epoch\'tan devam')
    n_egt = len(egt)
    agac_gor = torch.zeros(NTREE, dtype=torch.bool, device=DEV); agac_gor[ser_tree[R_ser[egt]]] = True
    seri_gor = torch.zeros(NSER, dtype=torch.bool, device=DEV); seri_gor[R_ser[egt]] = True
    for ep in range(bas_ep, AYAR['max_epoch']):
        t0 = time.time(); kar = egt[torch.randperm(n_egt, device=DEV)]; top = 0.
        for i in range(0, n_egt, AYAR['batch']):
            b = kar[i:i+AYAR['batch']]
            gz = (torch.rand(len(b), device=DEV) < AYAR['takson_dusur'])
            mu, ls, _ = model(b, gz)
            kayip_v = nll(mu, ls, hedef(b))
            kayip = kayip_v + (model.gz.ceza(seri_gor, agac_gor) / n_egt if AYAR['rastgele_etki'] else 0.)
            opt.zero_grad(set_to_none=True); kayip.backward()
            torch.nn.utils.clip_grad_norm_(ag, 5.0); opt.step(); top += kayip_v.item() * len(b)
        model.gz.gorgul_guncelle(seri_gor, agac_gor)
        mu, sg = tahmin(model, dog, seri_gor, agac_gor)
        yd = hedef(dog); v = nll(mu, torch.log(sg), yd).item()
        nz = yd > LOG_Y_SIFIR + 1e-6; v_rmse = torch.sqrt(((mu - yd)[nz] ** 2).mean()).item(); v_sg = torch.sqrt((sg ** 2).mean()).item()
        gecmis.append(dict(epoch=ep, egitim=top / n_egt, dogrulama=v, sn=time.time() - t0))
        ek = (f'  σ_alan {model.gz.sv_gorgul.item():.2f}  σ_ağaç {model.gz.su().item():.2f}' if AYAR['rastgele_etki'] else '')
        print(f'{is_ad} ep {ep:2d}  eğitim {top/n_egt:.4f}  doğrulama {v:.4f} (RMSE {v_rmse:.3f}, öngörü σ {v_sg:.3f}){ek}  ({time.time()-t0:.0f} sn)')
        if v < en_iyi - 1e-4:
            en_iyi, sabir = v, 0; torch.save(model.state_dict(), os.path.join(klasor, 'en_iyi.pt'))
        else:
            sabir += 1
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), epoch=ep, en_iyi=en_iyi, sabir=sabir, gecmis=gecmis), kayit)
        if sabir >= AYAR['sabir']: break
    # Test tahminleri (en iyi model)
    model.load_state_dict(torch.load(os.path.join(klasor, 'en_iyi.pt'), map_location=DEV))
    mu, sg = tahmin(model, test, seri_gor, agac_gor, gizle_alan=gizle)
    y = hedef(test)
    np.savez_compressed(os.path.join(klasor, 'test_tahmin.npz'), idx=test.cpu().numpy().astype(np.int32),
                        mu=mu.cpu().numpy().astype(np.float16), sigma=sg.cpu().numpy().astype(np.float16))
    nz = y > LOG_Y_SIFIR + 1e-6         # nokta ölçütleri sıfır olmayan halkalarda; NLL sansürlü olanları da içerir
    ozet = dict(is_ad=is_ad, n_egitim=n_egt, n_test=len(test), rmse_log=float(torch.sqrt(((mu - y)[nz] ** 2).mean())),
                bias_log=float((mu - y)[nz].mean()), nll=float(nll(mu, torch.log(sg), y)),
                kapsama90=float(((y - mu).abs() < 1.645 * sg)[nz].float().mean()), en_iyi_dogrulama=en_iyi,
                ag_param=ag_param_sayisi(model), gecmis=gecmis, ayar=AYAR)
    if not model_ad.startswith('serbest'):           # alan parametreleri (m, eşikler, ω) — şekiller için
        with torch.no_grad():
            p = model.parametreler(torch.arange(S, device=DEV), torch.full((S,), gizle, device=DEV) & t(site_takson, torch.bool))
        np.savez_compressed(os.path.join(klasor, 'alan_param.npz'), **{k: v.cpu().numpy() for k, v in p.items()},
                            r0_cm=F.softplus(model.gz.rho).detach().cpu().numpy(), u=model.gz.u.detach().cpu().numpy())
    json.dump(ozet, open(os.path.join(klasor, 'bitti.json'), 'w'), indent=1)
    os.remove(kayit)
    print(f'BİTTİ {is_ad}: test RMSE(log) {ozet["rmse_log"]:.3f}, kapsama %90 → {ozet["kapsama90"]:.3f}')

In [ ]:
# --- Ortak araçlar: topluluk tahmini, ölçütler, bootstrap ---
import glob, pandas as pd
from scipy.stats import norm
ANALIZ = os.path.join(ANA, 'ANALIZ_SON'); os.makedirs(ANALIZ, exist_ok=True)
Yn = hedef(torch.arange(NR, device=DEV)).cpu().numpy(); NZn = (R_w > 0).cpu().numpy()
SITE_n = ser_site[R_ser].cpu().numpy(); YIL_n = R_yil.cpu().numpy()
def topluluk(b, m):
    MU, V, idx = [], [], None
    for kl in sorted(glob.glob(os.path.join(SONUC, f'{b}__{m}__s*'))):
        if not os.path.exists(os.path.join(kl, 'bitti.json')): continue
        z = np.load(os.path.join(kl, 'test_tahmin.npz'))
        if idx is None: idx = z['idx']
        MU.append(z['mu'].astype(np.float32)); V.append(z['sigma'].astype(np.float32) ** 2)
    if not MU: return None
    MU, V = np.stack(MU), np.stack(V)
    return idx, MU.mean(0), np.sqrt(V.mean(0) + MU.var(0)), len(MU)
def olcutler(y, mu, sg, nz):
    r = (y - mu)[nz]; z = (y - mu) / sg
    nllv = np.where(nz, 0.5 * z**2 + np.log(sg) + 0.9189385, -norm.logcdf((math.log(0.01) - mu) / sg))
    crps = sg * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / math.sqrt(math.pi))
    return dict(RMSE=float(np.sqrt((r**2).mean())), bias=float(r.mean()), NLL=float(nllv.mean()),
                CRPS=float(crps[nz].mean()), kapsama90=float((np.abs(z) < 1.645)[nz].mean()))
rng = np.random.default_rng(2026)
def boot_fark(y, ma, mb, nz, grup, B=1000):
    """RMSE(a) − RMSE(b) ve grup (alan ya da yıl) bootstrap aralıkları"""
    g = grup[nz]; ea, eb = (y - ma)[nz] ** 2, (y - mb)[nz] ** 2
    u, inv = np.unique(g, return_inverse=True)
    Sa, Sb, n = np.bincount(inv, ea, len(u)), np.bincount(inv, eb, len(u)), np.bincount(inv, None, len(u))
    W = rng.multinomial(len(u), np.full(len(u), 1 / len(u)), size=B).astype(float)
    d = np.sqrt(W @ Sa / (W @ n)) - np.sqrt(W @ Sb / (W @ n))
    return float(np.sqrt(Sa.sum() / n.sum()) - np.sqrt(Sb.sum() / n.sum())), np.quantile(d, [0.025, 0.975]), np.quantile(d, [0.05, 0.95])
def alan_agirlikli_rmse(y, mu, nz, s):
    u, inv = np.unique(s[nz], return_inverse=True)
    return float(np.sqrt(np.bincount(inv, (y - mu)[nz] ** 2) / np.bincount(inv)).mean())

In [ ]:
# --- Senaryo ve özellik altyapısı (Defter 5–7 ile aynı) ---
from scipy.stats import spearmanr
import gc
CL = d['clim']; LAT = d['site_lat']
def gun_uzunlugu(lat):
    gun = np.array([15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349])
    dek = 23.45 * np.sin(np.deg2rad(360 * (284 + gun) / 365))
    x = -np.tan(np.deg2rad(lat))[:, None] * np.tan(np.deg2rad(dek))[None, :]
    return 24 / np.pi * np.arccos(np.clip(x, -1, 1))
def isi_indeksi(T):
    S_ = T.shape[0]; Tclim = np.nanmean(T.reshape(S_, -1, 12), axis=1)
    return np.sum(np.where(Tclim > 0, (np.maximum(Tclim, 0) / 5) ** 1.514, 0), axis=1)
def kova(T, P, lat, I, Mmax=0.76, Mmin=0.01, alph=0.093, m_th=4.886, mu_th=5.80, rootd=1000., M0=0.2):
    """Defter 1 ile aynı VS-Lite kova modeli; ısı indeksi I dışarıdan (taban iklim) verilir"""
    S_, N = T.shape; gu = gun_uzunlugu(lat)
    ngun = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]); L = (ngun / 30)[None, :] * (gu / 12)
    a = 6.75e-7 * I**3 - 7.71e-5 * I**2 + 1.792e-2 * I + 0.49
    M = np.empty((S_, N), np.float32); m = np.full(S_, M0)
    for t in range(N):
        Tt, Pt, Lt = T[:, t], P[:, t], L[:, t % 12]
        Ep = np.where(Tt <= 0, 0., np.where(Tt < 26.5, 16 * Lt * (10 * np.maximum(Tt, 0) / np.maximum(I, 1e-6)) ** a, -415.85 + 32.25 * Tt - 0.43 * Tt**2))
        Ep = np.maximum(Ep, 0); E = Ep * (m / Mmax)
        G_ = mu_th * alph / (1 + mu_th) * m * rootd; R_ = Pt * (m / Mmax) ** m_th + (alph / (1 + mu_th)) * m * rootd
        m = np.clip(m + (Pt - E - R_ - G_) / rootd, Mmin, Mmax); M[:, t] = m
    return M
es = lambda T: 0.6108 * np.exp(17.27 * T / (T + 237.3))
@torch.no_grad()
@torch.no_grad()
def tahmin_G(model, G, bs=65536):
    n = len(G['rp']); out = []
    for i in range(0, n, bs): out.append(model(None, None, G={k: v[i:i+bs] for k, v in G.items()})[0])
    return torch.cat(out)
def yukle(bolunme_ad, model_ad, tohum=0):
    kl = os.path.join(SONUC, f'{bolunme_ad.replace(":", "-")}__{model_ad}__s{tohum}')
    if not os.path.exists(os.path.join(kl, 'en_iyi.pt')): return None, None
    egt, _, test, _ = bolunme(bolunme_ad); kat_gor_ayarla(egt)
    m = yap(model_ad); m.load_state_dict(torch.load(os.path.join(kl, 'en_iyi.pt'), map_location=DEV)); m.eval()
    return m, test

import xgboost as xgb
print('xgboost', xgb.__version__)
NG_SEC = 2_000_000   # XGBoost eğitim alt örneklemi (halka); doğrulama 300 bin
def ozellik_ag(b, gizle):
    """[B, 126]: 24 ay × (T, logP, M, VPD) + log yarıçap + g_E ağırlıklı mevsim ortalamaları (önceki/bu yıl) + alan + cins, tür"""
    X, s = pencere(b); w = X[..., 4]
    mev = [((X[:, a:a+12, c] * w[:, a:a+12]).sum(1) / w[:, a:a+12].sum(1).clamp_min(1e-6))[:, None] for a in (0, 12) for c in range(4)]
    g = site_genus[s].float(); sp = site_species[s].float()
    if gizle: g = torch.full_like(g, float('nan')); sp = torch.full_like(sp, float('nan'))
    return torch.cat([X[..., :4].flatten(1), torch.log(R_rp[b] + 0.1)[:, None]] + mev + [DESC[s], ENV[s], g[:, None], sp[:, None]], 1)
def ozellik_np(idx, gizle, bs=500_000):
    return np.concatenate([ozellik_ag(idx[i:i+bs], gizle).cpu().numpy() for i in range(0, len(idx), bs)]).astype(np.float32)
FT = ['q'] * (96 + 1 + 8 + DESC.shape[1] + ENV.shape[1]) + ['c', 'c']
XDEV = 'cuda' if DEV == 'cuda' else 'cpu'
def kaydet(b, ad, idx, mu, sg, ek):
    kl = os.path.join(SONUC, f'{b.replace(":", "-")}__{ad}__s0'); os.makedirs(kl, exist_ok=True)
    np.savez_compressed(os.path.join(kl, 'test_tahmin.npz'), idx=idx.astype(np.int32), mu=mu.astype(np.float16), sigma=sg.astype(np.float16))
    json.dump(ek, open(os.path.join(kl, 'bitti.json'), 'w'))
def bitti(b, ad): return os.path.exists(os.path.join(SONUC, f'{b.replace(":", "-")}__{ad}__s0', 'bitti.json'))

def dkm_ozellik(b, gizle, g_gor):
    """Doğrusal karma model tasarımı: sabit, log r, (log r)², 8 mevsim iklimi, DESC, ENV, cins (gizlide görülen cinslerin ortalaması)"""
    X, s = pencere(b); w = X[..., 4]
    mev = [((X[:, a:a+12, c] * w[:, a:a+12]).sum(1) / w[:, a:a+12].sum(1).clamp_min(1e-6))[:, None] for a in (0, 12) for c in range(4)]
    lr = torch.log(R_rp[b] + 0.1)[:, None]
    oh = F.one_hot(site_genus[s], NG + 1).float()
    if gizle: oh = g_gor.float()[None, :].expand(len(b), -1) / g_gor.sum()
    return torch.cat([torch.ones_like(lr), lr, lr ** 2] + mev + [DESC[s], ENV[s], oh], 1), s

In [ ]:
# --- Klasik modeller: yeniden kullanılabilir kurulum ve tahmin fonksiyonları ---
XGB_VARSAYILAN = dict(max_depth=10, eta=0.08, subsample=0.8, colsample_bytree=0.6, max_bin=128, min_child_weight=50)
def xgb_kur(b, prm=XGB_VARSAYILAN, n_egt=NG_SEC, n_tur=2000, nicel=True, tohum=0):
    egt, dog, test, gizle = bolunme(b)
    egt = egt[R_w[egt] > 0]; dog = dog[R_w[dog] > 0]
    g = torch.Generator(device=DEV).manual_seed(tohum)
    egt = egt[torch.randperm(len(egt), device=DEV, generator=g)[:n_egt]]; dog = dog[torch.randperm(len(dog), device=DEV, generator=g)[:300_000]]
    if AYAR['hizli_deneme']: egt, dog, test, n_tur = egt[:100_000], dog[:20_000], test[:20_000], 40
    dtr = xgb.DMatrix(ozellik_np(egt, False), hedef(egt).cpu().numpy(), feature_types=FT, enable_categorical=True)
    dva = xgb.DMatrix(ozellik_np(dog, False), hedef(dog).cpu().numpy(), feature_types=FT, enable_categorical=True)
    ort = dict(device=XDEV, tree_method='hist', max_cat_to_onehot=1, seed=tohum, **prm)
    m1 = xgb.train(dict(ort, objective='reg:squarederror'), dtr, n_tur, evals=[(dva, 'dog')], early_stopping_rounds=50, verbose_eval=False)
    m2 = None
    if nicel:
        m2 = xgb.train(dict(ort, objective='reg:quantileerror', quantile_alpha=np.array([0.05, 0.95])), dtr, min(n_tur, 600),
                       evals=[(dva, 'dog')], early_stopping_rounds=50, verbose_eval=False)
    v = float(np.sqrt(np.mean((m1.predict(dva, iteration_range=(0, m1.best_iteration + 1)) - dva.get_label()) ** 2)))
    del dtr, dva
    return dict(m1=m1, m2=m2, test=test, gizle=gizle, dog_rmse=v)
import gc
def xgb_tahmin(K, idx, gizle=False):
    MU, SG = [], []
    for i in range(0, len(idx), 1_000_000):
        dte = xgb.DMatrix(ozellik_np(idx[i:i+1_000_000], gizle), feature_types=FT, enable_categorical=True)
        MU.append(K['m1'].predict(dte, iteration_range=(0, K['m1'].best_iteration + 1)))
        if K['m2'] is not None:
            q = K['m2'].predict(dte, iteration_range=(0, K['m2'].best_iteration + 1)); SG.append(np.maximum((q[:, 1] - q[:, 0]) / 3.29, 0.05))
    return np.concatenate(MU), (np.concatenate(SG) if SG else None)
def dkm_kur(b, bs=400_000):
    egt, dog, test, gizle = bolunme(b); egt = egt[R_w[egt] > 0]
    if AYAR['hizli_deneme']: egt, test = egt[:200_000], test[:20_000]
    g_gor = torch.zeros(NG + 1, dtype=torch.bool, device=DEV); g_gor[site_genus[ser_site[R_ser[egt]]]] = True
    y_all = hedef(egt); site_e = ser_site[R_ser[egt]]; u = torch.zeros(S, device=DEV)
    for it in range(3):
        XtX = 0; Xty = 0
        for i in range(0, len(egt), bs):
            Z, s = dkm_ozellik(egt[i:i+bs], False, g_gor); rr = y_all[i:i+bs] - u[s]
            XtX = XtX + Z.T.double() @ Z.double(); Xty = Xty + Z.T.double() @ rr.double()
        p_ = XtX.shape[0]; beta = torch.linalg.solve(XtX + 1e-3 * torch.eye(p_, device=DEV, dtype=torch.float64) * XtX.diagonal().mean(), Xty).float()
        res = torch.cat([y_all[i:i+bs] - dkm_ozellik(egt[i:i+bs], False, g_gor)[0] @ beta for i in range(0, len(egt), bs)])
        n_s = torch.bincount(site_e, minlength=S).float(); top = torch.bincount(site_e, res, minlength=S)
        gor = n_s > 0; rbar = top[gor] / n_s[gor]
        s2e = float(((res - (top / n_s.clamp_min(1))[site_e]) ** 2).sum() / (len(res) - gor.sum()))
        s2u = max(float(rbar.var() - (s2e / n_s[gor]).mean()), 1e-4)
        u = torch.zeros(S, device=DEV); u[gor] = n_s[gor] * s2u / (n_s[gor] * s2u + s2e) * rbar
    return dict(beta=beta, u=u, n_s=n_s, g_gor=g_gor, s2e=s2e, s2u=s2u, test=test, gizle=gizle)
@torch.no_grad()
def dkm_tahmin(K, idx, gizle=False, bs=400_000):
    MU = []
    for i in range(0, len(idx), bs):
        Z, s = dkm_ozellik(idx[i:i+bs], gizle, K['g_gor']); MU.append((Z @ K['beta'] + K['u'][s]).cpu().numpy())
    return np.concatenate(MU)

In [ ]:
# --- Ortam bilgisi: kullanılan sürümler (yeniden üretilebilirlik) ---
import sys, platform, importlib
def surum(p):
    try: return importlib.import_module(p).__version__
    except Exception: return 'yok'
SURUM = {p: surum(p) for p in ['numpy', 'pandas', 'scipy', 'torch', 'xgboost', 'xarray', 'zarr', 'fsspec', 'gcsfs']}
SURUM.update(python=platform.python_version(), cuda=(torch.version.cuda if torch.cuda.is_available() else 'yok'))
print(SURUM); json.dump(SURUM, open(os.path.join(ANALIZ, 'I0_surumler.json'), 'w'), indent=1)

In [ ]:
# --- I1a. BAĞIMSIZ DUYARLILIK TESTİ — eğitim: 10° blokların 5 katı; her kat dışarıda bırakılarak U+S, T+S (3 tohum), ayarlı XGBoost, karma model ---
# Her alan tam olarak bir katın test kümesinde: model o alanın HİÇBİR halkasını eğitimde görmez.
# Kesilirse yeniden "Tümünü çalıştır": biten eğitimler atlanır.
BLOK = [f'blok:{k}' for k in sorted(np.unique(site_kat).tolist())]
N_TOHUM_B1 = 1 if AYAR['hizli_deneme'] else 3
yol_ayar = os.path.join(ANALIZ, 'G4_xgb_ayar_arama.csv')
if os.path.exists(yol_ayar):
    en = pd.read_csv(yol_ayar).sort_values('dog_rmse').iloc[0]
    PRM = {k: (int(en[k]) if k in ('max_depth', 'max_bin') else float(en[k])) for k in ['max_depth', 'eta', 'subsample', 'colsample_bytree', 'max_bin', 'min_child_weight', 'reg_lambda'] if k in en}
else:
    PRM = dict(XGB_VARSAYILAN)
print('Katlar:', BLOK, '| XGBoost ayarları:', PRM)
t_bas = time.time()
for tohum in range(N_TOHUM_B1):                       # önce tüm katlar tohum 0 ile: kesilse bile analiz yapılabilir
    for b in BLOK:
        for m in ['teori_ortam', 'serbest_ortam']:
            egit(b, m, tohum); gc.collect(); torch.cuda.empty_cache() if DEV == 'cuda' else None
    print(f'--- tohum {tohum} bitti ({(time.time() - t_bas) / 60:.0f} dk) ---')
for b in BLOK:
    if not bitti(b, 'xgb_ayar'):
        t0 = time.time(); K_ = xgb_kur(b, PRM, n_egt=NG_SEC, n_tur=4000, nicel=False)
        mu, _ = xgb_tahmin(K_, K_['test'])
        kaydet(b, 'xgb_ayar', K_['test'].cpu().numpy(), mu, np.full_like(mu, 0.5), dict(prm=PRM, tur=int(K_['m1'].best_iteration), sn=time.time() - t0))
        print('XGBoost bitti:', b, f'{(time.time() - t0) / 60:.0f} dk'); del K_; gc.collect()
    if not bitti(b, 'dkm'):
        t0 = time.time(); K_ = dkm_kur(b); mu = dkm_tahmin(K_, K_['test'])
        kaydet(b, 'dkm', K_['test'].cpu().numpy(), mu, np.full_like(mu, math.sqrt(K_['s2e'] + K_['s2u'])), dict(s2e=K_['s2e'], s2u=K_['s2u'], sn=time.time() - t0))
        print('Karma model bitti:', b); del K_; gc.collect()
print(f'I1a eğitimleri tamam ({(time.time() - t_bas) / 3600:.1f} saat)')

In [ ]:
# --- I1b. Bağımsız duyarlılık: gözlenen ve model duyarlılığı AYNI işlemle ---
# Gözlenen: log halka genişliği → seri içi 31 yıllık hareketli ortalamadan sapma → alan-yıl ortalaması (kronoloji)
#           → g_E ağırlıklı yıllık sıcaklık ve log yağış anomalilerine çoklu regresyon (Defter 7, G2 ile aynı).
# Model:    AYNI halkalar için modelin (alanı hiç görmemiş) tahmin ettiği log halka genişliğine birebir aynı işlem.
# Anlamlılık: artıkların 1. derece otokorelasyonuna göre düzeltilmiş t (|t_düz| > 2); ayrıca düzeltmesiz.
# Güven aralıkları: alanların bootstrap ile yeniden örneklenmesi (2000 tekrar).
B1M = {'serbest_ortam': 'U+S', 'teori_ortam': 'T+S', 'xgb_ayar': 'XGBoost', 'dkm': 'Karma model'}
R_w_n = R_w.cpu().numpy(); R_ser_n = R_ser.cpu().numpy()
PRED = {}
for m in B1M:
    a = np.full(NR, np.nan, np.float32); n_ok = 0
    for b in BLOK:
        r_ = topluluk(b.replace(':', '-'), m)
        if r_ is None: continue
        a[r_[0]] = r_[1]; n_ok += 1
    if n_ok == len(BLOK): PRED[m] = a
    print(f'{B1M[m]}: {n_ok}/{len(BLOK)} kat' + ('' if n_ok == len(BLOK) else '  → eksik, analiz dışı'))
ortak = R_w_n > 0
for v in PRED.values(): ortak &= np.isfinite(v)
print(f'Ortak halka: {ortak.sum():,}')

def kronoloji(y, mask):
    ser = R_ser_n[mask]; yil = YIL_n[mask]; s = SITE_n[mask]; yv = y[mask].astype(np.float64)
    o = np.lexsort((yil, ser)); ser, yil, s, yv = ser[o], yil[o], s[o], yv[o]
    n_ = len(yv); pos = np.arange(n_)
    bas_ = np.r_[0, np.flatnonzero(np.diff(ser)) + 1]; uzun = np.diff(np.r_[bas_, n_])
    sb = np.repeat(bas_, uzun); se_ = sb + np.repeat(uzun, uzun)
    lo = np.maximum(sb, pos - 15); hi = np.minimum(se_, pos + 16)
    cs = np.r_[0, np.cumsum(yv)]; say = hi - lo
    iz = np.where(say >= 15, yv - (cs[hi] - cs[lo]) / np.maximum(say, 1), np.nan); k_ = np.isfinite(iz)
    return pd.DataFrame(dict(s=s[k_], yil=yil[k_], iz=iz[k_])).groupby(['s', 'yil']).iz.mean()
w_ = OZ[:, :, 4].reshape(S, NY, 12); Tm_ = OZ[:, :, 0].reshape(S, NY, 12); Pm_ = OZ[:, :, 1].reshape(S, NY, 12)
Tg = ((Tm_ * w_).sum(2) / w_.sum(2).clamp_min(1e-6)).cpu().numpy(); Pg = np.log(np.expm1(Pm_.cpu().numpy()).sum(2) + 1)
del w_, Tm_, Pm_
def sapma(A):
    k_ = np.ones(31) / 31; n = np.convolve(np.ones(A.shape[1]), k_, mode='same'); out = np.full_like(A, np.nan)
    for i in range(A.shape[0]): out[i] = A[i] - np.convolve(A[i], k_, mode='same') / n
    return out
Ta, Pa = sapma(Tg), sapma(Pg)
def alan_reg(g):
    bos = pd.Series(dict(bT=np.nan, bP=np.nan, tT=np.nan, tP=np.nan, tT_duz=np.nan, tP_duz=np.nan, r1=np.nan, n=len(g)))
    g = g.dropna()
    if len(g) < 30 or g.Ta.std() < 1e-6 or g.Pa.std() < 1e-6: return bos
    X = np.c_[np.ones(len(g)), g.Ta, g.Pa]; y = g.iz.values
    if np.linalg.cond(X.T @ X) > 1e10: return bos
    XtX = np.linalg.inv(X.T @ X); b = XtX @ X.T @ y; r = y - X @ b; s2 = r @ r / (len(g) - 3); se = np.sqrt(np.diag(XtX) * s2)
    r1 = float(np.corrcoef(r[:-1], r[1:])[0, 1]) if r.std() > 0 else 0.0
    f = math.sqrt((1 + r1) / (1 - r1)) if 0 < r1 < 0.99 else 1.0      # etkin örneklem düzeltmesi (AR(1))
    return pd.Series(dict(bT=b[1], bP=b[2], tT=b[1] / se[1], tP=b[2] / se[2], tT_duz=b[1] / se[1] / f, tP_duz=b[2] / se[2] / f, r1=r1, n=len(g)))
def duyarlilik(kron):
    df = kron.reset_index(); df['Ta'] = Ta[df.s.values, df.yil.values - YIL0]; df['Pa'] = Pa[df.s.values, df.yil.values - YIL0]
    return df.groupby('s')[['iz', 'Ta', 'Pa']].apply(alan_reg).dropna()
GOZ = duyarlilik(kronoloji(np.log(np.maximum(R_w_n, 1e-6)), ortak))
JJ = GOZ.add_suffix('_g')
for m in PRED:
    JJ = JJ.join(duyarlilik(kronoloji(PRED[m], ortak))[['bT', 'bP']].add_suffix('_' + m), how='inner')
JJ['kat'] = site_kat[JJ.index]; JJ['kurak'] = -desc[JJ.index, 5]; JJ['soguk'] = -desc[JJ.index, 2]
JJ.to_csv(os.path.join(ANALIZ, 'I1c_alan_duyarliliklari.csv'))
print(f'Duyarlılık hesaplanan alan: {len(JJ)} | anlamlı sıcaklık (düzeltmesiz / düzeltilmiş): {(JJ.tT_g.abs() > 2).sum()} / {(JJ.tT_duz_g.abs() > 2).sum()}'
      f' | yağış: {(JJ.tP_g.abs() > 2).sum()} / {(JJ.tP_duz_g.abs() > 2).sum()}')

rngB = np.random.default_rng(2027); NB_ = 2000
def sira(x): return pd.Series(x).rank().values
def olc_boot(W, bo, bm, sig):
    """W: [B, n] alan ağırlıkları. Dönüş: yön uyumu (anlamlı alanlarda), Spearman (sabit sıralar üzerinde ağırlıklı Pearson), kalibrasyon eğimi"""
    uy = (np.sign(bo) == np.sign(bm)).astype(float)
    Ws = W[:, sig]; yon = (Ws @ uy[sig]) / np.maximum(Ws.sum(1), 1)
    def wpear(x, y):
        n = W.sum(1); mx = W @ x / n; my = W @ y / n
        cxy = W @ (x * y) / n - mx * my; vx = W @ (x * x) / n - mx ** 2; vy = W @ (y * y) / n - my ** 2
        return cxy / np.sqrt(vx * vy), cxy / vy
    rho, _ = wpear(sira(bo), sira(bm)); _, egim = wpear(bo, bm)
    return dict(yon_uyumu=yon, spearman=rho, kalibrasyon_egimi=egim)
I1, I1F = [], []
for kume, sec_ in [('tüm alanlar', np.ones(len(JJ), bool)), ('en soğuk çeyrek', JJ.soguk.values >= np.quantile(JJ.soguk, 0.75)),
                   ('en kurak çeyrek', JJ.kurak.values >= np.quantile(JJ.kurak, 0.75))]:
    jj = JJ[sec_]; n = len(jj)
    W = np.vstack([np.ones(n), rngB.multinomial(n, np.full(n, 1 / n), size=NB_)]).astype(float)   # satır 0 = asıl örneklem
    for deg, b_, t_ in [('sıcaklık', 'bT', 'tT_duz'), ('yağış', 'bP', 'tP_duz')]:
        sig = jj[t_ + '_g'].abs().values > 2; bo = jj[b_ + '_g'].values; sonuc = {}
        for m in PRED:
            sonuc[m] = olc_boot(W, bo, jj[f'{b_}_{m}'].values, sig)
            for olc, v in sonuc[m].items():
                I1.append(dict(kume=kume, degisken=deg, model=B1M[m], olcut=olc, deger=float(v[0]), alt95=float(np.nanquantile(v[1:], 0.025)),
                               ust95=float(np.nanquantile(v[1:], 0.975)), n_alan=n, n_anlamli=int(sig.sum()),
                               gozlenen_pozitif_orani=float((bo[sig] > 0).mean()) if sig.any() else np.nan))
        if 'teori_ortam' in sonuc:
            for m in PRED:
                if m == 'teori_ortam': continue
                for olc in sonuc[m]:
                    dfr = sonuc['teori_ortam'][olc] - sonuc[m][olc]
                    I1F.append(dict(kume=kume, degisken=deg, karsilastirma=f'T+S − {B1M[m]}', olcut=olc, fark=float(dfr[0]),
                                    alt95=float(np.nanquantile(dfr[1:], 0.025)), ust95=float(np.nanquantile(dfr[1:], 0.975)),
                                    T_S_ustun_bootstrap_orani=float(np.nanmean(dfr[1:] > 0))))
I1 = pd.DataFrame(I1); I1.to_csv(os.path.join(ANALIZ, 'I1_bagimsiz_duyarlilik.csv'), index=False)
I1F = pd.DataFrame(I1F); I1F.to_csv(os.path.join(ANALIZ, 'I1b_bagimsiz_duyarlilik_farklar.csv'), index=False)
print(I1.pivot_table(index=['kume', 'degisken', 'olcut'], columns='model', values='deger').round(3).to_string())
print(I1F[I1F.olcut != 'kalibrasyon_egimi'].round(3).to_string(index=False))
# Bloklar dışarıda bırakıldığında doğruluk (mekânsal aktarım): ortak halkalarda
YY = Yn[ortak]; I1d = []
for m in PRED:
    e_ = PRED[m][ortak] - YY
    I1d.append(dict(model=B1M[m], RMSE_log=float(np.sqrt((e_ ** 2).mean())), bias_log=float(e_.mean()),
                    alan_agirlikli_RMSE=alan_agirlikli_rmse(Yn[ortak], PRED[m][ortak], np.ones(ortak.sum(), bool), SITE_n[ortak])))
I1d = pd.DataFrame(I1d); I1d.to_csv(os.path.join(ANALIZ, 'I1d_blok_dogruluk.csv'), index=False); print(I1d.round(3).to_string(index=False))

In [ ]:
# --- I2. Tablo 2 için güven aralıkları: senaryo tepkileri (Defter 7 G1 ile aynı 50.000 halka ve modeller), alan bootstrap ---
gen = torch.Generator(device='cpu').manual_seed(0)
_, _, test_z, _ = bolunme('zaman')
sec = test_z[torch.randperm(len(test_z), generator=gen)[:50000].to(DEV)]
siteler = torch.unique(ser_site[R_ser[sec]]).cpu().numpy()
T0 = CL[siteler, :, :, 0].reshape(len(siteler), -1).astype(np.float64); P0 = CL[siteler, :, :, 1].reshape(len(siteler), -1).astype(np.float64)
V0 = CL[siteler, :, :, 3].reshape(len(siteler), -1).astype(np.float64); I0 = isi_indeksi(T0)
M_taban = kova(T0, P0, LAT[siteler], I0); M_yagis = kova(T0, P0 * 1.2, LAT[siteler], I0); M_sicak = kova(T0 + 1, P0, LAT[siteler], I0)
V_sicak = np.maximum(es(T0 + 1) - (es(T0) - V0), 0)
OZ_yedek = OZ[torch.as_tensor(siteler, device=DEV)].clone(); si = torch.as_tensor(siteler, device=DEV)
def ayarla(T=None, P=None, M=None, V=None):
    OZ[si] = OZ_yedek
    if T is not None: OZ[si, :, 0] = t(T, torch.float32)
    if P is not None: OZ[si, :, 1] = t(np.log1p(np.maximum(P, 0)), torch.float32)
    if M is not None: OZ[si, :, 2] = t(M, torch.float32)
    if V is not None: OZ[si, :, 3] = t(V, torch.float32)
s_sec = ser_site[R_ser[sec]].cpu().numpy()
kurak_q = -desc[s_sec, 5] >= np.quantile(-desc[s_sec, 5], 0.75); soguk_q = -desc[s_sec, 2] >= np.quantile(-desc[s_sec, 2], 0.75)
yol_i2 = os.path.join(ANALIZ, 'I2_senaryo_halka.npz'); DEG = dict(np.load(yol_i2)) if os.path.exists(yol_i2) else {}
I2M = {'xgb': 'XGBoost', 'dkm': 'Karma model', 'serbest': 'U', 'serbest_ortam': 'U+S', 'teori': 'T', 'teori_vpd': 'T_D',
       'teori_ortam': 'T+S', 'teori_vpd_ortam': 'T_D+S', 'teori_uyum_ortam': 'T+S uyumlu'}
try:
    for m_ad in [m for m in I2M if m not in ('xgb', 'dkm')]:
        if f'{m_ad}|dP' in DEG: continue
        Y0, Y1, Y2 = [], [], []
        for tohum in range(5):
            model, _ = yukle('zaman', m_ad, tohum)
            if model is None: continue
            ayarla(M=M_taban); Y0.append(tahmin_G(model, girdiler(sec)).cpu().numpy())
            ayarla(P=P0 * 1.2, M=M_yagis); Y1.append(tahmin_G(model, girdiler(sec)).cpu().numpy())
            ayarla(T=T0 + 1, M=M_sicak, V=V_sicak); Y2.append(tahmin_G(model, girdiler(sec)).cpu().numpy())
            del model
        ayarla()
        if Y0: DEG[f'{m_ad}|dP'] = np.mean(Y1, 0) - np.mean(Y0, 0); DEG[f'{m_ad}|dT'] = np.mean(Y2, 0) - np.mean(Y0, 0); np.savez_compressed(yol_i2, **DEG)
    if 'xgb|dP' not in DEG:
        kat_gor_ayarla(bolunme('zaman')[0]); KX0 = xgb_kur('zaman', nicel=False); KD0 = dkm_kur('zaman')
        for ad, f in [('xgb', lambda: xgb_tahmin(KX0, sec)[0]), ('dkm', lambda: dkm_tahmin(KD0, sec))]:
            ayarla(M=M_taban); y0 = f(); ayarla(P=P0 * 1.2, M=M_yagis); y1 = f(); ayarla(T=T0 + 1, M=M_sicak, V=V_sicak); y2 = f()
            DEG[f'{ad}|dP'] = y1 - y0; DEG[f'{ad}|dT'] = y2 - y0
        del KX0, KD0; gc.collect(); np.savez_compressed(yol_i2, **DEG)
finally:
    ayarla()                                           # iklim tensörü her durumda özgün hâline döner
u_s, inv_s = np.unique(s_sec, return_inverse=True); nS_ = len(u_s)
W2 = np.vstack([np.ones(nS_), rngB.multinomial(nS_, np.full(nS_, 1 / nS_), size=NB_)]).astype(float)
def oran_boot(gosterge, alt):
    pay = np.bincount(inv_s, gosterge & alt, nS_); payda = np.bincount(inv_s, alt, nS_)
    return (W2 @ pay) / np.maximum(W2 @ payda, 1)
I2 = []
for m, ad in I2M.items():
    if f'{m}|dP' not in DEG: continue
    dP, dT = DEG[f'{m}|dP'], DEG[f'{m}|dT']; hep = np.ones(len(dP), bool)
    for olc, gos, alt in [('yağış +%20 → azalan', dP < -1e-3, hep), ('ısınma → azalan (en soğuk çeyrek)', dT < -1e-3, soguk_q),
                          ('ısınma → artan (en soğuk çeyrek)', dT > 1e-3, soguk_q), ('ısınma → azalan (en kurak çeyrek)', dT < -1e-3, kurak_q),
                          ('ısınma → azalan (tümü)', dT < -1e-3, hep)]:
        v = 100 * oran_boot(gos, alt)
        I2.append(dict(model=ad, olcut=olc, yuzde=float(v[0]), alt95=float(np.quantile(v[1:], 0.025)), ust95=float(np.quantile(v[1:], 0.975))))
I2 = pd.DataFrame(I2); I2.to_csv(os.path.join(ANALIZ, 'I2_senaryo_guven_araliklari.csv'), index=False)
print(I2.assign(aralik=lambda x: x.yuzde.round(1).astype(str) + ' (' + x.alt95.round(1).astype(str) + '–' + x.ust95.round(1).astype(str) + ')')
        .pivot_table(index='model', columns='olcut', values='aralik', aggfunc='first').to_string())

In [ ]:
# --- I3. Toplanmış ölçekte R² (zamansal test 1961–2024; topluluk tahminleri) ---
# Halka; ağaç × on yıl; seri ortalaması; alan kronolojisi (alan × yıl); alan ortalaması (alanlar arası); alan içi yıllar arası
R2M = {'serbest_ortam': 'U+S', 'teori_ortam': 'T+S', 'xgb_ayar': 'XGBoost', 'dkm': 'Karma model'}
agac_n_all = ser_tree.cpu().numpy()
r2 = lambda a, b: float(1 - ((a - b) ** 2).sum() / ((a - a.mean()) ** 2).sum())
I3 = []
for m, ad in R2M.items():
    r_ = topluluk('zaman', m)
    if r_ is None: print('yok:', m); continue
    idx = r_[0]; k_ = NZn[idx]; idx, mu = idx[k_], r_[1][k_]
    df = pd.DataFrame(dict(y=Yn[idx], mu=mu, s=SITE_n[idx], yil=YIL_n[idx], ser=R_ser_n[idx], agac=agac_n_all[R_ser_n[idx]]))
    sat = dict(model=ad, halka=r2(df.y.values, df.mu.values))
    for adi, grp in [('agac_on_yil', ['agac', df.yil // 10]), ('seri_ortalamasi', ['ser']), ('alan_kronolojisi', ['s', 'yil']), ('alan_ortalamasi', ['s'])]:
        g = df.groupby(grp)[['y', 'mu']].mean(); sat[adi] = r2(g.y.values, g.mu.values)
    kr = df.groupby(['s', 'yil'])[['y', 'mu']].mean(); kr = kr - kr.groupby(level=0).transform('mean')
    sat['alan_ici_yillar_arasi'] = r2(kr.y.values, kr.mu.values)
    sat['kalibre_halka'] = float(np.corrcoef(df.y, df.mu)[0, 1] ** 2)
    I3.append(sat)
I3 = pd.DataFrame(I3); I3.to_csv(os.path.join(ANALIZ, 'I3_toplanmis_R2.csv'), index=False); print(I3.round(3).to_string(index=False))

In [ ]:
# --- I4–I6 hazırlık: son modeller, CMIP6 değişimleri (Defter 8'de indirildi), gelecek iklimi, ağaç örneklemi ---
kat_gor_ayarla(bolunme('tum')[0])
yol_xgb = os.path.join(SONUC, 'tum__xgb_model.json'); yol_dkm = os.path.join(SONUC, 'tum__dkm.pt')
KX = dict(m1=xgb.Booster(), m2=None); KX['m1'].load_model(yol_xgb); KX['tur'] = int(json.load(open(yol_xgb + '.tur'))['tur'])
KD = torch.load(yol_dkm, map_location=DEV)
GCM = ['GFDL-ESM4', 'IPSL-CM6A-LR', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'UKESM1-0-LL']
SSP = ['ssp126', 'ssp245', 'ssp585']; DEG3 = ['tas', 'pr', 'hurs']
KLS = os.path.join(ANA, 'CMIP6')
LATs, LONs = d['site_lat'].astype(float), d['site_lon'].astype(float)
MEVCUT = [g for g in GCM if all(os.path.exists(os.path.join(KLS, f'{g}_{e}.npz')) for e in ['historical'] + SSP)]
print('Kullanılabilir iklim modelleri:', MEVCUT)
GE = d['gE'].astype(np.float32)
R0 = 1955 - YIL0; NY60 = 60
T_b = CL[:, R0:R0 + NY60, :, 0].astype(np.float64); P_b = CL[:, R0:R0 + NY60, :, 1].astype(np.float64); V_b = CL[:, R0:R0 + NY60, :, 3].astype(np.float64)
I_b = isi_indeksi(T_b.reshape(S, -1))
def kova2(T, P, lat, I, Mmax=0.76, Mmin=0.01, alph=0.093, m_th=4.886, mu_th=5.80, rootd=1000., M0=0.2):
    S_, N = T.shape; gu = gun_uzunlugu(lat)
    ngun = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]); L = (ngun / 30)[None, :] * (gu / 12)
    a = 6.75e-7 * I**3 - 7.71e-5 * I**2 + 1.792e-2 * I + 0.49
    M = np.empty((S_, N), np.float32); PE = np.empty((S_, N), np.float32); m = np.full(S_, M0)
    for t_ in range(N):
        Tt, Pt, Lt = T[:, t_], P[:, t_], L[:, t_ % 12]
        Ep = np.where(Tt <= 0, 0., np.where(Tt < 26.5, 16 * Lt * (10 * np.maximum(Tt, 0) / np.maximum(I, 1e-6)) ** a, -415.85 + 32.25 * Tt - 0.43 * Tt**2))
        Ep = np.maximum(Ep, 0); E = Ep * (m / Mmax)
        G_ = mu_th * alph / (1 + mu_th) * m * rootd; R_ = Pt * (m / Mmax) ** m_th + (alph / (1 + mu_th)) * m * rootd
        m = np.clip(m + (Pt - E - R_ - G_) / rootd, Mmin, Mmax); M[:, t_] = m; PE[:, t_] = Ep
    return M, PE
def iklim(dT=None, rP=None, dRH=None):
    T, P, V = T_b.copy(), P_b.copy(), V_b.copy()
    if dT is not None:
        esT = es(T); RH = np.clip((esT - V) / np.maximum(esT, 1e-6), 0.02, 1.0)
        T = T + dT.T[:, None, :]; P = P * rP.T[:, None, :]; RH = np.clip(RH + dRH.T[:, None, :] / 100, 0.02, 1.0); V = es(T) * (1 - RH)
    M, PE = kova2(T.reshape(S, -1), P.reshape(S, -1), LATs, I_b)
    X = np.stack([T.reshape(S, -1), np.log1p(np.maximum(P.reshape(S, -1), 0)), M, V.reshape(S, -1), np.tile(GE, (1, NY60))], -1).astype(np.float32)
    son = slice(30 * 12, 60 * 12)
    Pa_ = P.reshape(S, -1)[:, son].reshape(S, 30, 12).sum(2).mean(1); PEa = PE[:, son].reshape(S, 30, 12).sum(2).mean(1)
    nor = dict(T=T.reshape(S, -1)[:, son].mean(1), logP=np.log1p(Pa_), logPET=np.log1p(PEa), kur=Pa_ / np.maximum(PEa, 1), V=V.reshape(S, -1)[:, son].mean(1), M=M[:, son].mean(1))
    return X, nor
def deltalar(g, e, y0, y1, ham=False):
    h = np.load(os.path.join(KLS, f'{g}_historical.npz')); f = np.load(os.path.join(KLS, f'{g}_{e}.npz'))
    hb = {v: np.nanmean(h[v], 0) for v in DEG3}; fb = {v: np.nanmean(f[v][y0 - 2015:y1 - 2015 + 1], 0) for v in DEG3}
    oran = fb['pr'] / np.maximum(hb['pr'], 1e-12)
    if ham: return oran
    return fb['tas'] - hb['tas'], np.clip(oran, 0.3, 3.0), fb['hurs'] - hb['hurs']
X_taban, NOR_taban = iklim()
# Ağaç örneklemi (Defter 8 ile aynı: aynı tohum, aynı seçim)
sira_ = np.lexsort((YIL_n, R_ser_n)); son_i = sira_[np.r_[np.flatnonzero(np.diff(R_ser_n[sira_])), len(sira_) - 1]]
son_i = son_i[YIL_n[son_i] >= 1995]
agac_n = ser_tree.cpu().numpy()[R_ser_n[son_i]]
_, ilk = np.unique(agac_n, return_index=True); son_i = son_i[ilk]
rr = np.random.default_rng(11); son_i = rr.permutation(son_i)
alan_say = {}; sec_i = []
for i_ in son_i:
    s_ = SITE_n[i_]
    if alan_say.get(s_, 0) < 4: sec_i.append(i_); alan_say[s_] = alan_say.get(s_, 0) + 1
N_AGAC = 3000 if AYAR['hizli_deneme'] else 12000
sec_i = np.array(sec_i[:N_AGAC])
R_rp_n = R_rp.cpu().numpy()
A_s = SITE_n[sec_i]; A_ser = R_ser_n[sec_i]; A_r = R_rp_n[sec_i] + R_w_n[sec_i] / 10
At_s = t(A_s, torch.long); At_ser = t(A_ser, torch.long); At_r = t(A_r, torch.float32)
print(f'{len(sec_i)} ağaç, {len(np.unique(A_s))} alan')
DESC_ham = desc.astype(np.float64); D_ort, D_ss = DESC_ham.mean(0), DESC_ham.std(0)
def desc_guncel(nor_f):
    D = DESC_ham.copy()
    D[:, 2] += nor_f['T'] - NOR_taban['T']; D[:, 3] += nor_f['logP'] - NOR_taban['logP']; D[:, 4] += nor_f['logPET'] - NOR_taban['logPET']
    D[:, 5] *= nor_f['kur'] / np.maximum(NOR_taban['kur'], 1e-6); D[:, 6] += nor_f['V'] - NOR_taban['V']; D[:, 7] += nor_f['M'] - NOR_taban['M']
    return t((D - D_ort) / D_ss, torch.float32), t(MBAR.cpu().numpy() + (nor_f['M'] - NOR_taban['M']), torch.float32), D
def pencere_X(XA, j): return XA[:, (j - 1) * 12:(j + 1) * 12]
def G_kur(X, DS=None, MB=None):
    G = alan_girdi(At_s)
    if DS is not None: G['desc'] = DS[At_s]
    if MB is not None: G['mbar'] = MB[At_s]
    G.update(X=X, rp=At_r, ser=At_ser); return G
def oz_X(X, DS=None):
    w = X[..., 4]; s = At_s
    mev = [((X[:, a:a+12, c] * w[:, a:a+12]).sum(1) / w[:, a:a+12].sum(1).clamp_min(1e-6))[:, None] for a in (0, 12) for c in range(4)]
    D = DESC[s] if DS is None else DS[s]
    return torch.cat([X[..., :4].flatten(1), torch.log(At_r + 0.1)[:, None]] + mev + [D, ENV[s], site_genus[s].float()[:, None], site_species[s].float()[:, None]], 1), mev, D
@torch.no_grad()
def dkm_X(X, DS=None):
    _, mev, D = oz_X(X, DS); lr = torch.log(At_r + 0.1)[:, None]
    Z = torch.cat([torch.ones_like(lr), lr, lr ** 2] + mev + [D, ENV[At_s], F.one_hot(site_genus[At_s], NG + 1).float()], 1)
    return (Z @ KD['beta'] + KD['u'][At_s]).cpu().numpy()
def xgb_X(X, DS=None):
    Z = oz_X(X, DS)[0].cpu().numpy(); dm = xgb.DMatrix(Z, feature_types=FT, enable_categorical=True)
    return KX['m1'].predict(dm, iteration_range=(0, KX['tur'] + 1))
AGLAR = {}
for m in ['serbest_ortam', 'teori_ortam', 'teori_uyum_ortam']:
    AGLAR[m] = [y_ for y_ in (yukle('tum', m, s_)[0] for s_ in range(5)) if y_ is not None]
    print(m, len(AGLAR[m]), 'tohum')
kat_gor_ayarla(bolunme('tum')[0])
YILLAR = range(31, 60)
def ortalama_uye(Xs, DS=None, MB=None, klasik=False):
    """29 yılın ortalama log tahmini; sinir ağlarında HER TOHUM AYRI (m#k)"""
    XA = Xs[At_s]; top = None
    for j in YILLAR:
        X = pencere_X(XA, j); G = G_kur(X, DS, MB); o = {}
        for m, L in AGLAR.items():
            for k_, mm_ in enumerate(L): o[f'{m}#{k_}'] = tahmin_G(mm_, G).cpu().numpy()
        if klasik: o['xgb'] = xgb_X(X, DS); o['dkm'] = dkm_X(X, DS)
        top = o if top is None else {k: top[k] + o[k] for k in o}
    return {k: v / len(YILLAR) for k, v in top.items()}
NORM6 = DESC_ham[:, 2:]; mu6 = NORM6.mean(0); Ci6 = np.linalg.inv(np.cov(NORM6, rowvar=False) + 1e-9 * np.eye(6))
dm6 = lambda Z: np.sqrt(np.einsum('ij,jk,ik->i', Z - mu6, Ci6, Z - mu6)); q99 = np.quantile(dm6(NORM6), 0.99)

In [ ]:
# --- I4. Projeksiyonlarda topluluk (tohum) belirsizliği: her iklim modeli × her tohum ayrı; yağış oranı sınırına takılma ---
Xt_taban = t(X_taban, torch.float32); BAZ_U = ortalama_uye(Xt_taban)
yol_i4 = os.path.join(ANALIZ, 'I4_uye_projeksiyon.npz'); UYE = dict(np.load(yol_i4)) if os.path.exists(yol_i4) else {}
SINIR = []
for g in MEVCUT:
    for e in SSP:
        for (y0, y1) in [(2041, 2070), (2071, 2100)]:
            oran = deltalar(g, e, y0, y1, ham=True)
            SINIR.append(dict(gcm=g, ssp=e, donem=f'{y0}–{y1}', ay_alan_sinir_disi=float(((oran < 0.3) | (oran > 3)).mean()),
                              agac_alanlarinda=float(((oran[:, A_s] < 0.3) | (oran[:, A_s] > 3)).mean())))
            anahtar0 = f'{g}|{e}|{y0}'
            if all(any(k.startswith(f'{anahtar0}|{nv}|') for k in UYE) for nv in ('sabit', 'guncel')): continue
            dT, rP, dRH = deltalar(g, e, y0, y1); Xf, nor_f = iklim(dT, rP, dRH); Xt_f = t(Xf, torch.float32); DS, MB, Dh = desc_guncel(nor_f)
            for nv, (ds_, mb_) in [('sabit', (None, None)), ('guncel', (DS, MB))]:
                F_ = ortalama_uye(Xt_f, ds_, mb_)
                for k in F_: UYE[f'{anahtar0}|{nv}|{k}'] = (F_[k] - BAZ_U[k]).astype(np.float16)
            np.savez_compressed(yol_i4, **UYE); print('tamam:', g, e, y0)
SINIR = pd.DataFrame(SINIR); SINIR.to_csv(os.path.join(ANALIZ, 'I4c_yagis_orani_siniri.csv'), index=False)
print('Yağış oranı 0,3–3 dışında kalan ay×alan oranı: en çok', round(100 * SINIR.ay_alan_sinir_disi.max(), 3), '%; ağaç alanlarında en çok',
      round(100 * SINIR.agac_alanlarinda.max(), 3), '%')
MODEL_AD = {'serbest_ortam': 'U+S', 'teori_ortam': 'T+S', 'teori_uyum_ortam': 'T+S uyumlu'}
I4 = []
for e in SSP:
    for y0 in ['2041', '2071']:
        for nv in ['sabit', 'guncel']:
            for m, ad in MODEL_AD.items():
                M_ = np.full((len(MEVCUT), len(AGLAR[m])), np.nan)
                for i, g in enumerate(MEVCUT):
                    for k_ in range(len(AGLAR[m])):
                        v = UYE.get(f'{g}|{e}|{y0}|{nv}|{m}#{k_}')
                        if v is not None: M_[i, k_] = np.median(100 * (np.exp(v.astype(np.float32)) - 1))
                tum_ort = np.mean([np.mean([UYE[f'{g}|{e}|{y0}|{nv}|{m}#{k_}'].astype(np.float32) for k_ in range(len(AGLAR[m]))], 0) for g in MEVCUT], 0)
                I4.append(dict(ssp=e, donem=y0, normaller=nv, model=ad, medyan_degisim_topluluk=float(np.median(100 * (np.exp(tum_ort) - 1))),
                               gcm_x_tohum_medyani=float(np.nanmedian(M_)), alt5=float(np.nanquantile(M_, 0.05)), ust95=float(np.nanquantile(M_, 0.95)),
                               gcm_arasi_min=float(np.nanmin(np.nanmean(M_, 1))), gcm_arasi_max=float(np.nanmax(np.nanmean(M_, 1))),
                               tohum_arasi_ss_ort=float(np.nanmean(np.nanstd(M_, 1))), gcm_arasi_ss=float(np.nanstd(np.nanmean(M_, 1)))))
I4 = pd.DataFrame(I4); I4.to_csv(os.path.join(ANALIZ, 'I4_projeksiyon_belirsizlik.csv'), index=False)
print(I4[I4.donem == '2071'].round(2).to_string(index=False))

In [ ]:
# --- I5. B_max kontrolü: gözlenen halkalar, bugünkü ağaçlar ve 2025–2100 serbest yürütme sonunda B > B_max olan ağaç oranı ---
with torch.no_grad():
    PP = [mm_.parametreler(torch.arange(S, device=DEV)) for mm_ in AGLAR['teori_ortam']]
Bmax_s = np.median([p['Bmax'].cpu().numpy() for p in PP], 0); r0_s = np.median([p['r0'].cpu().numpy() for p in PP], 0)
m_s = np.median([p['m'].cpu().numpy() for p in PP], 0); eta_s = np.median([p['eta'].cpu().numpy() for p in PP], 0)
def G_pot(B, eta, Bm, m): return eta * B ** m * np.log1p(np.exp(np.minimum(20 * (1 - (B / Bm) ** (1 - m)), 50))) / 20
Bgrid = np.exp(np.linspace(np.log(1), np.log(5e5), 400))[None, :]
Gmax = G_pot(Bgrid, eta_s[:, None], Bmax_s[:, None], m_s[:, None]).max(1)
oran_Bmax = G_pot(Bmax_s, eta_s, Bmax_s, m_s) / Gmax
B_halka = math.pi * (r0_s[SITE_n] + R_rp_n + R_w_n / 10) ** 2
I5 = [dict(olcut='B_max eşdeğer yarıçap (cm), alanlar medyanı', deger=float(np.median(np.sqrt(Bmax_s / math.pi)))),
      dict(olcut='B_max eşdeğer yarıçap (cm), alanların %5–95 aralığı', deger=f'{np.quantile(np.sqrt(Bmax_s / math.pi), 0.05):.1f}–{np.quantile(np.sqrt(Bmax_s / math.pi), 0.95):.1f}'),
      dict(olcut="G(B_max) / en büyük G, alanlar medyanı", deger=float(np.median(oran_Bmax))),
      dict(olcut='Gözlenen halkalarda B > B_max oranı', deger=float((B_halka > Bmax_s[SITE_n]).mean())),
      dict(olcut='Bugünkü ağaçlarda (projeksiyon örneklemi) B > B_max oranı', deger=float((math.pi * (r0_s[A_s] + A_r) ** 2 > Bmax_s[A_s]).mean()))]
yol_h5 = os.path.join(ANALIZ, 'H5_serbest_yurutme.npz')
if os.path.exists(yol_h5):
    SY = dict(np.load(yol_h5))
    for k_ in [k for k in SY if k.endswith('|teori_ortam|r')]:
        I5.append(dict(olcut=f'Serbest yürütme 2100 sonunda B > B_max oranı: {k_.replace("|teori_ortam|r", "")}',
                       deger=float((math.pi * (r0_s[A_s] + SY[k_]) ** 2 > Bmax_s[A_s]).mean())))
I5 = pd.DataFrame(I5); I5.to_csv(os.path.join(ANALIZ, 'I5_Bmax_kontrol.csv'), index=False); print(I5.to_string(index=False))

In [ ]:
# --- I6. Uygulama (GitHub) dosyaları: alan tablosu, T+S parametreleri, iklim normalleri, tepki ızgarası, projeksiyonlar, ONNX modelleri ---
# Ham halka ölçümü YOK: yalnız alan bilgileri (NOAA'da açık), modelden türetilmiş değerler ve atıflı iklim/toprak özetleri.
UYG = os.path.join(ANA, 'UYGULAMA_VERI'); os.makedirs(UYG, exist_ok=True)
soz_desc = soz.get('desc_vars', [f'desc{i}' for i in range(desc.shape[1])]); soz_env = soz.get('env_vars', [f'env{i}' for i in range(env.shape[1])])
n_seri = np.bincount(ser_site.cpu().numpy(), minlength=S)
ilk_yil = pd.Series(YIL_n).groupby(SITE_n).min().reindex(range(S)).values; son_yil = pd.Series(YIL_n).groupby(SITE_n).max().reindex(range(S)).values
r_ref = pd.Series(A_r).groupby(A_s).median().reindex(range(S)).fillna(15.0).clip(3, 60).values      # tepki ızgarası için alan başına referans yarıçap (cm)
alan_tab = pd.DataFrame(dict(i=np.arange(S), kod=soz['site_code'], enlem=LATs, boylam=LONs, cins=np.array(soz['genus'])[site_genus.cpu().numpy()],
                             tur=np.array(soz['species'])[site_species.cpu().numpy()], bolge=soz['site_bolge'], seri=n_seri, ilk_yil=ilk_yil, son_yil=son_yil,
                             r_ref_cm=r_ref, Bmax_cm2=Bmax_s, m=m_s, r0_cm=r0_s))
for j, ad in enumerate(soz_desc): alan_tab[f'd_{ad}'] = desc[:, j]
for j, ad in enumerate(soz_env): alan_tab[f'e_{ad}'] = env[:, j]
alan_tab['biyom'] = d['site_biyom']; alan_tab['ekobolge'] = d['site_eko']; alan_tab['koppen'] = d['site_kg']
alan_tab.to_csv(os.path.join(UYG, 'alanlar.csv'), index=False)
# T+S parametreleri: [tohum, alan, P] float32; sıra meta.json'da
P_SIRA = ['eta', 'Bmax', 'm', 'aT', 'cT', 'sT', 'aM', 'cM', 'sM', 'omega', 'log_sigma', 'r0']
PAR = np.stack([np.concatenate([p[k].reshape(S, -1).cpu().numpy() for k in P_SIRA], 1) for p in PP]).astype(np.float32)
PAR.tofile(os.path.join(UYG, 'ts_param.f32'))
# İklim normalleri 1985–2014: aylık T (°C), P (mm), M (toprak nemi), VPD (kPa), g_E
nk = slice(1985 - YIL0, 2015 - YIL0)
NORM = np.stack([CL[:, nk, :, 0].mean(1), CL[:, nk, :, 1].mean(1), soil[:, nk].mean(1), CL[:, nk, :, 3].mean(1), d['gE']], -1).astype(np.float32)
NORM.tofile(os.path.join(UYG, 'iklim_normal.f32'))
# Tepki ızgarası: tüm modeller, alan başına referans yarıçapta, 1985–2014 iklimine düzgün ΔT ve yağış oranı (toprak nemi, PET, VPD yeniden hesaplı)
DT_IZ = [-2, -1, 0, 1, 2, 3, 4]; RP_IZ = [0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4]
yol_iz = os.path.join(ANALIZ, 'I6_tepki_izgarasi.npz')
if not os.path.exists(yol_iz):
    yedek = (At_s, At_ser, At_r)
    ilk_ser = np.full(S, -1); ss_ = ser_site.cpu().numpy(); ilk_ser[ss_[::-1]] = np.arange(len(ss_))[::-1]
    At_s = t(np.arange(S), torch.long); At_ser = t(ilk_ser, torch.long); At_r = t(r_ref, torch.float32)
    IZ = np.zeros((5, len(DT_IZ), len(RP_IZ), S), np.float32)
    try:
        for a_, dT_ in enumerate(DT_IZ):
            for b_, rP_ in enumerate(RP_IZ):
                Xf, _ = iklim(np.full((12, S), float(dT_)), np.full((12, S), float(rP_)), np.zeros((12, S)))
                F_ = ortalama_uye(t(Xf, torch.float32), klasik=True)
                for c_, m in enumerate(['serbest_ortam', 'teori_ortam', 'teori_uyum_ortam']):
                    IZ[c_, a_, b_] = np.mean([F_[f'{m}#{k}'] for k in range(len(AGLAR[m]))], 0)
                IZ[3, a_, b_] = F_['xgb']; IZ[4, a_, b_] = F_['dkm']
            print('ızgara ΔT =', dT_)
    finally:
        At_s, At_ser, At_r = yedek
    np.savez_compressed(yol_iz, IZ=IZ, dT=DT_IZ, rP=RP_IZ)
IZ = np.load(yol_iz)['IZ']
i0, j0 = DT_IZ.index(0), RP_IZ.index(1.0)
(100 * (np.exp(IZ - IZ[:, i0:i0 + 1, j0:j0 + 1]) - 1)).astype(np.float32).tofile(os.path.join(UYG, 'tepki_izgarasi.f32'))   # % değişim [model, ΔT, oran, alan]
# Projeksiyonlar (Defter 8, H1): alan düzeyi, iklim modelleri ortalaması, % değişim
yol_h1 = os.path.join(ANALIZ, 'H1_sabit_boyut_agac.npz')
if os.path.exists(yol_h1):
    H1 = dict(np.load(yol_h1)); NOVd = dict(np.load(os.path.join(ANALIZ, 'H1_yenilik_alan.npz')))
    gcm_h1 = sorted({k.split('|')[0] for k in H1}); pr_ = {}
    for e in SSP:
        for y0 in ['2041', '2071']:
            for nv in ['sabit', 'guncel']:
                for m in ['serbest_ortam', 'teori_ortam', 'teori_uyum_ortam', 'xgb', 'dkm']:
                    L = [H1[f'{g}|{e}|{y0}|{nv}|{m}'] for g in gcm_h1 if f'{g}|{e}|{y0}|{nv}|{m}' in H1]
                    if L: pr_[f'{e}_{y0}_{nv}_{m}'] = pd.Series(100 * (np.exp(np.mean(L, 0)) - 1)).groupby(A_s).mean()
            pr_[f'{e}_{y0}_yenilik'] = pd.Series(np.mean([NOVd[f'{g}|{e}|{y0}'] for g in gcm_h1 if f'{g}|{e}|{y0}' in NOVd], 0)[np.unique(A_s)], index=np.unique(A_s))
    pd.DataFrame(pr_).round(2).rename_axis('i').to_csv(os.path.join(UYG, 'projeksiyon_alan.csv'))
# Karma model katsayıları (doğrusal: tarayıcıda doğrudan hesaplanabilir)
json.dump(dict(beta=KD['beta'].cpu().numpy().tolist(), u_alan=KD['u'].cpu().numpy().tolist(), s2e=KD['s2e'], s2u=KD['s2u'],
               tasarim='[1, log(r+0.1), log(r+0.1)^2, 8 g_E ağırlıklı mevsim ortalaması (önceki yıl T,logP,M,VPD; bu yıl T,logP,M,VPD), DESC(std), ENV(std), cins one-hot (NG+1)]'),
          open(os.path.join(UYG, 'karma_model.json'), 'w'))
# ONNX: T+S parametre ağı (alan bilgisi → büyüme parametreleri) ve U+S ağı (iklim penceresi + alan → log halka genişliği), her tohum
import subprocess
for paket in ['onnx', 'onnxruntime']:
    try: importlib.import_module(paket)
    except ImportError: subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', paket], check=False)
importlib.invalidate_caches(); import onnxruntime as ort
class TSParam(nn.Module):
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, g, sp, desc_, env_, b, e, k):
        p = s.m.parametreler(dict(g=g, sp=sp, desc=desc_, env=env_, b=b, e=e, k=k, mbar=desc_[:, 0]))
        return torch.cat([p[q].reshape(len(g), -1) for q in P_SIRA], 1)
class USAg(nn.Module):
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, X, rp, g, sp, desc_, env_, b, e, k):
        m = s.m; h = m.conv(((X - oz_ort) / oz_ss).transpose(1, 2)).flatten(1)
        o = m.mlp(torch.cat([h, m.ge(g), m.se(sp), desc_, torch.log1p(rp)[:, None], m.ortam(dict(env=env_, b=b, e=e, k=k))], 1))
        return torch.stack([o[:, 0] - 0.3, o[:, 1] - 0.3], 1)
def onnx_yaz(mod, girdi, adlar, cikis, yol):
    dyn = {a: {0: 'n'} for a in adlar + [cikis]}
    try:
        torch.onnx.export(mod, girdi, yol, input_names=adlar, output_names=[cikis], dynamic_axes=dyn, opset_version=17, dynamo=False)
    except Exception as h_:
        print('eski dışa aktarıcı çalışmadı, dynamo deneniyor:', type(h_).__name__)
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'onnxscript'], check=False)
        torch.onnx.export(mod, girdi, yol, input_names=adlar, output_names=[cikis], dynamic_axes=dyn, dynamo=True)
n8 = 8; s8 = torch.arange(n8, device=DEV)
alan8 = (site_genus[s8], site_species[s8], DESC[s8], ENV[s8], site_biyom[s8], site_eko[s8], site_kg[s8])
ad_alan = ['cins', 'tur', 'desc', 'env', 'biyom', 'ekobolge', 'koppen']
ONNX_KONTROL = []
for k_, mm_ in enumerate(AGLAR['teori_ortam']):
    mod = TSParam(mm_).eval(); yol = os.path.join(UYG, f'ts_param_s{k_}.onnx'); onnx_yaz(mod, alan8, ad_alan, 'param', yol)
    with torch.no_grad(): ref = mod(*alan8).cpu().numpy()
    oo = ort.InferenceSession(yol).run(None, {a: v.cpu().numpy() for a, v in zip(ad_alan, alan8)})[0]
    ONNX_KONTROL.append(dict(model=f'T+S parametre ağı s{k_}', en_buyuk_fark=float(np.abs(oo - ref).max())))
X8 = OZ[s8, 600:624]; rp8 = torch.full((n8,), 10.0, device=DEV)
for k_, mm_ in enumerate(AGLAR['serbest_ortam']):
    mod = USAg(mm_).eval(); yol = os.path.join(UYG, f'us_s{k_}.onnx'); girdi = (X8, rp8) + alan8
    onnx_yaz(mod, girdi, ['X', 'rp_cm'] + ad_alan, 'mu_logsigma', yol)
    with torch.no_grad(): ref = mod(*girdi).cpu().numpy()
    oo = ort.InferenceSession(yol).run(None, {a: v.cpu().numpy() for a, v in zip(['X', 'rp_cm'] + ad_alan, girdi)})[0]
    ONNX_KONTROL.append(dict(model=f'U+S ağı s{k_}', en_buyuk_fark=float(np.abs(oo - ref).max())))
print(pd.DataFrame(ONNX_KONTROL).to_string(index=False))
J_ = AYAR['J']
meta = dict(
    surum='Defter 9', tarih=time.strftime('%Y-%m-%d'), S=S, J=J_, n_tohum_ts=len(AGLAR['teori_ortam']),
    ts_param=dict(dosya='ts_param.f32', bicim='float32 [tohum, alan, P] (little-endian)', sira=P_SIRA,
                  boyut={'eta': 1, 'Bmax': 1, 'm': 1, 'aT': J_, 'cT': J_, 'sT': J_, 'aM': J_, 'cM': J_, 'sM': J_, 'omega': 24, 'log_sigma': 1, 'r0': 1}),
    teori_ileri=('r: yarıçap (cm); B = π r² (cm²); x = (B/Bmax)^(1−m); G = η B^m softplus(20(1−x))/20; '
                 'g_T = Σ_j w_j σ((T−cT_j)/sT_j) / Σ_j w_j, w_j = softplus(aT_j) + 1e-4 (g_M aynı, M ile); '
                 'C = Σ_{24 ay} ω g_E min(g_T, g_M); BAI = G·C; halka (mm) = (√((B+BAI)/π) − r)·10. '
                 'Pencere: önceki yıl Ocak – bu yıl Aralık (24 ay); kanallar T (°C), log(1+P mm), M, VPD (kPa), g_E.'),
    iklim_normal=dict(dosya='iklim_normal.f32', bicim='float32 [alan, 12 ay, 5]: T °C, P mm, M, VPD kPa, g_E', donem='1985–2014', kaynak='CRU TS 4.09 (OGL v3)'),
    tepki_izgarasi=dict(dosya='tepki_izgarasi.f32', bicim='float32 [5 model, 7 ΔT, 9 yağış oranı, alan], % değişim (ΔT=0, oran=1 tabanına göre)',
                        modeller=['U+S', 'T+S', 'T+S uyumlu', 'XGBoost', 'Karma model'], dT=DT_IZ, rP=RP_IZ, yaricap='alanlar.csv: r_ref_cm'),
    standart=dict(desc_ort=desc.mean(0).tolist(), desc_ss=desc.std(0).tolist(), env_ort=env.mean(0).tolist(), env_ss=env.std(0).tolist(),
                  desc_ad=soz_desc, env_ad=soz_env, oz_ort=oz_ort.cpu().numpy().tolist(), oz_ss=oz_ss.cpu().numpy().tolist()),
    siniflar=dict(cins=soz['genus'], tur=soz['species'], NB=NB, NE=NE, NK=NK, bilinmeyen_cins=NG, bilinmeyen_tur=NS),
    onnx=dict(ts_param='ts_param_s{k}.onnx: girdiler cins, tur (int64), desc, env (standart), biyom, ekobolge, koppen (int64) → param [n, P] (sıra yukarıda)',
              us='us_s{k}.onnx: X [n,24,5] (ham kanallar), rp_cm [n], + alan girdileri → [μ log mm, log σ]', kontrol=ONNX_KONTROL),
    lisans=dict(kod='MIT', model_ve_sonuclar='CC BY 4.0', iklim='CRU TS 4.09 — Open Government Licence v3 (atıf: Harris ve ark. 2020)',
                cmip6='CMIP6 — CC BY 4.0 (iklim modeli grupları; Eyring ve ark. 2016)', toprak_ortam='SoilGrids, RESOLVE Ecoregions, Köppen–Geiger — CC BY 4.0',
                itrdb='Ham halka ölçümleri yeniden dağıtılmaz; alan kodları NOAA NCEI ITRDB kayıtlarına bağlantıdır.'),
    surumler=SURUM)
json.dump(meta, open(os.path.join(UYG, 'meta.json'), 'w'), ensure_ascii=False, indent=1)
print('Uygulama dosyaları:', sorted(os.listdir(UYG)))
print('==== DEFTER 9 TAMAM ====  Tablolar:', ANALIZ, '| Uygulama:', UYG)